# 🚀 Session 5: Model Tracking with MLflow (Intermediate)

## MLOps with Agentic AI - Advanced Certification Course

---

### 👋 Welcome!

In this session, you'll **master advanced MLflow tracking** to become proficient in experiment management, model versioning, and professional ML workflow organization.

### 🎯 Learning Objectives

By the end of this session, you will:

✅ **Master advanced MLflow Tracking API**
- Use autolog() vs manual logging strategically
- Log complex artifacts (plots, datasets, pipelines)
- Create nested runs for hyperparameter tuning
- Implement parent-child run hierarchies

✅ **Navigate MLflow UI like a pro**
- Use advanced filtering and search
- Compare multiple runs side-by-side
- Create parallel coordinates plots
- Organize experiments with tags

✅ **Apply production-grade patterns**
- Track data lineage and versioning
- Implement team collaboration workflows
- Ensure experiment reproducibility
- Prepare models for registry (Session 6)

---

### 📚 Session Roadmap

| Part | Topic | Time | Focus |
|------|-------|------|-------|
| 0 | Setup & Verification | 10 min | Starter kit check |
| 1 | MLflow Components & Tracking Foundations | 25 min | Understanding the ecosystem |
| 2 | Autolog vs Manual Logging | 25 min | Strategic logging approaches |
| 3 | Advanced Artifacts & Dataset Versioning | 30 min | Production tracking patterns |
| 4 | Hyperparameter Tuning & Comparison | 30 min | Nested runs, grid search |
| 5 | MLflow UI Mastery | 30 min | Advanced navigation & filtering |
| 6 | Production Patterns & Best Practices | 30 min | Real-world workflows |

**Total Duration**: ~3 hours

---

### 🔗 Connection to Other Sessions

**📌 Session 3 (Foundation)**
- We learned: Basic `mlflow.start_run()`, simple param/metric logging
- Today: Advanced patterns, complex artifacts, nested runs

**📌 Session 4 (Git & Reproducibility)**
- We learned: Version control, code organization
- Today: Track Git commits in MLflow for full reproducibility

**📌 Session 6 (Next Week - Model Registry)**
- Preview: Take our tracked experiments and promote to production
- We'll use the best models from today's session!

---

### 💡 Teaching Philosophy

**90% MLflow Tracking | 10% Model Building**

This session uses pre-built models from the starter kit so you can focus on **mastering MLflow**, not debugging code. This is how you'll work in production: using existing models and focusing on tracking, monitoring, and lifecycle management.

Let's begin! 🎉

---

# Part 0: Setup & Verification

⏱️ **Time**: 10 minutes

Let's ensure everything is ready before we start tracking experiments.

In [ ]:
# Cell 1: Environment verification

import sys
import subprocess

print("🔍 Verifying Session 5 Environment...\n")

# Check Python version
python_version = sys.version_info
print(f"✅ Python Version: {python_version.major}.{python_version.minor}.{python_version.micro}")

if python_version.major < 3 or (python_version.major == 3 and python_version.minor < 9):
    print("⚠️  Warning: Python 3.9+ recommended for best compatibility")

# Check required packages
required_packages = [
    'mlflow',
    'numpy',
    'pandas',
    'sklearn',
    'matplotlib',
    'xgboost',
    'lightgbm'
]

print("\n📦 Checking Required Packages...")
missing_packages = []

for package in required_packages:
    try:
        __import__(package.replace('-', '_'))
        print(f"   ✅ {package}")
    except ImportError:
        print(f"   ❌ {package} - NOT INSTALLED")
        missing_packages.append(package)

if missing_packages:
    print(f"\n⚠️  Missing packages: {', '.join(missing_packages)}")
    print("   Run: pip install -r requirements.txt")
else:
    print("\n✅ All packages installed!")

# Verify starter kit structure
import os

print("\n📁 Verifying Starter Kit Structure...")
required_dirs = ['utils', 'models', 'data', 'experiments']
for dir_name in required_dirs:
    if os.path.exists(dir_name):
        print(f"   ✅ {dir_name}/")
    else:
        print(f"   ⚠️  {dir_name}/ - Missing (make sure you're in the starter kit directory)")

print("\n✅ Environment check complete!")

In [ ]:
# Cell 2: Import all starter kit utilities

print("📥 Importing Starter Kit Modules...\n")

# Data utilities
try:
    from utils.data_loader import load_customer_churn_data, load_iris_data
    print("✅ Data loaders imported")
except ImportError as e:
    print(f"❌ Data loaders failed: {e}")

# Preprocessing utilities
try:
    from utils.preprocessor import scale_features, DataPreprocessor, encode_categorical_features
    print("✅ Preprocessing tools imported")
except ImportError as e:
    print(f"❌ Preprocessing tools failed: {e}")

# Evaluation utilities
try:
    from utils.evaluator import evaluate_classifier, print_evaluation_report
    print("✅ Evaluation tools imported")
except ImportError as e:
    print(f"❌ Evaluation tools failed: {e}")

# Visualization utilities
try:
    from utils.visualizer import (
        plot_confusion_matrix,
        plot_roc_curve,
        plot_feature_importance,
        plot_learning_curve,
        plot_precision_recall_curve
    )
    print("✅ Visualization tools imported")
except ImportError as e:
    print(f"❌ Visualization tools failed: {e}")

# MLflow helpers
try:
    from utils.mlflow_helpers import (
        log_metrics_dict,
        log_params_dict,
        set_tags_from_dict,
        get_or_create_experiment
    )
    print("✅ MLflow helpers imported")
except ImportError as e:
    print(f"❌ MLflow helpers failed: {e}")

# Model modules
try:
    from models.sklearn_models import train_random_forest, train_logistic_regression
    from models.xgboost_models import train_xgboost_classifier
    from models.lightgbm_models import train_lightgbm_classifier
    from models.keras_models import train_shallow_nn
    from models.model_configs import (
        RANDOM_FOREST_CONFIGS,
        XGBOOST_CONFIGS,
        LIGHTGBM_CONFIGS
    )
    print("✅ Model training functions imported")
except ImportError as e:
    print(f"❌ Model functions failed: {e}")

# Standard imports
import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

print("\n✅ All imports successful!\n")
print("💡 Tip: If any imports failed, check that you're in the starter kit directory")

In [ ]:
# Cell 3: Configure MLflow

print("⚙️  Configuring MLflow...\n")

# Set tracking URI (local by default)
mlflow.set_tracking_uri("./mlruns")
print(f"📍 Tracking URI: {mlflow.get_tracking_uri()}")

# Create/get our experiment
experiment_name = "Session5_Advanced_Tracking"
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    experiment_id = mlflow.create_experiment(
        experiment_name,
        tags={
            "course": "MLOps with Agentic AI",
            "session": "5",
            "topic": "Advanced Tracking"
        }
    )
    print(f"✅ Created new experiment: {experiment_name}")
else:
    experiment_id = experiment.experiment_id
    print(f"✅ Using existing experiment: {experiment_name}")

# Set as active experiment
mlflow.set_experiment(experiment_name)
print(f"🎯 Active experiment: {experiment_name} (ID: {experiment_id})")

# Verify connection
try:
    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment(experiment_id)
    print(f"\n✅ MLflow connection verified!")
    print(f"   Experiment lifecycle: {exp.lifecycle_stage}")
except Exception as e:
    print(f"\n❌ Connection failed: {e}")
    print("   Try running: python setup_mlflow.py")

In [ ]:
# Cell 4: Load and verify datasets

print("📊 Loading Datasets...\n")

# Load customer churn dataset (our primary dataset)
X_train, X_test, y_train, y_test = load_customer_churn_data(test_size=0.2, random_state=42)

print("✅ Customer Churn Dataset:")
print(f"   - Training samples: {len(X_train):,}")
print(f"   - Test samples: {len(X_test):,}")
print(f"   - Features: {X_train.shape[1]}")
print(f"   - Churn rate (train): {y_train.mean():.2%}")
print(f"   - Churn rate (test): {y_test.mean():.2%}\n")

print("📋 Feature Names:")
print(f"   {', '.join(X_train.columns.tolist()[:10])}...\n")

print("✅ Data loaded and ready for experimentation!")
print("\n💡 Note: We'll use this dataset throughout the session for all experiments.")

### 🎯 Launch MLflow UI

Before we proceed, let's start the **MLflow UI** so you can view your experiments in real-time.

**Open a new terminal** and run:

```bash
mlflow ui
```

Then open your browser to: **http://127.0.0.1:5000**

**What you should see:**
- List of experiments in the left sidebar
- "Session5_Advanced_Tracking" experiment (we just created it)
- Clean interface ready for tracking

**💡 Pro Tip**: Keep the MLflow UI open in a separate browser tab throughout this session. You'll be switching back and forth frequently!

**⚠️ Troubleshooting**:
- Port 5000 busy? Use: `mlflow ui --port 5001`
- Can't find mlruns? Make sure you're in the starter kit directory

---

✅ **Setup Complete!** Let's start tracking experiments!

---

# Part 1: MLflow Components Overview & Tracking Foundations

⏱️ **Time**: 25 minutes

## Understanding the MLflow Ecosystem

Before diving deep into tracking, let's understand **where tracking fits** in the broader MLflow ecosystem.

### 📦 The 4 Components of MLflow

MLflow consists of **4 main components**:

```
┌─────────────────────────────────────────────────────────────┐
│                        MLflow Platform                      │
├─────────────┬─────────────┬─────────────┬───────────────────┤
│  TRACKING   │  PROJECTS   │   MODELS    │     REGISTRY      │
│  (Session 5)│ (Session 6) │ (Session 6) │   (Session 6)     │
└─────────────┴─────────────┴─────────────┴───────────────────┘
```

Let's understand each component:

---

### 1. 📊 MLflow Tracking (TODAY - Session 5)

**Purpose**: Record and query experiments

**What you track**:
- **Parameters**: Hyperparameters (learning_rate, max_depth)
- **Metrics**: Performance measures (accuracy, F1, AUC)
- **Artifacts**: Files (models, plots, datasets)
- **Metadata**: Tags, notes, source code

**Why it matters**:
- "Which experiment had accuracy > 0.90?"
- "What hyperparameters did I use 2 months ago?"
- "Can I reproduce this result?"

**Today's focus**: Master this component completely!

---

### 2. 📦 MLflow Projects (Session 6)

**Purpose**: Package ML code in reusable, reproducible format

**What it does**:
- Packages code + dependencies
- Defines entry points
- Enables reproducibility

**Example**:
```yaml
# MLproject file
name: customer_churn_model
entry_points:
  main:
    parameters:
      n_estimators: {type: int, default: 100}
    command: "python train.py --n_estimators {n_estimators}"
```

**Session 6 preview**: We'll convert today's experiments into MLflow Projects

---

### 3. 🎁 MLflow Models (Session 6)

**Purpose**: Standard format for packaging ML models

**What it provides**:
- Model abstraction (works with sklearn, TensorFlow, PyTorch, etc.)
- Deployment flexibility (REST API, batch, streaming)
- Signature and schema enforcement

**Example**:
```python
# Save model in MLflow format
mlflow.sklearn.log_model(model, "model")

# Load and use anywhere
loaded_model = mlflow.pyfunc.load_model(model_uri)
predictions = loaded_model.predict(data)
```

---

### 4. 📋 MLflow Model Registry (Session 6)

**Purpose**: Centralized model store with lifecycle management

**Lifecycle stages**:
1. **None**: Just trained, not registered
2. **Staging**: Being tested/validated
3. **Production**: Serving live traffic
4. **Archived**: Retired from production

**Example workflow**:
```python
# Register a model from today's experiments
mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="customer_churn_classifier"
)

# Promote to production
client.transition_model_version_stage(
    name="customer_churn_classifier",
    version=3,
    stage="Production"
)
```

**Session 6 preview**: We'll take today's best models and promote them through the registry!

---

### 🔄 How They Work Together

```
Session 5 (Today):          Session 6 (Next Week):
┌──────────────┐            ┌──────────────┐
│   TRACKING   │───────────>│   PROJECTS   │
│ Experiments  │            │  Packaging   │
└──────────────┘            └──────────────┘
       │                            │
       │                            ▼
       │                    ┌──────────────┐
       └───────────────────>│    MODELS    │
                            │  Deployment  │
                            └──────────────┘
                                    │
                                    ▼
                            ┌──────────────┐
                            │   REGISTRY   │
                            │  Lifecycle   │
                            └──────────────┘
```

**💡 Key Insight**: Today we focus on **Tracking** - the foundation. Everything else builds on top of good tracking!

### 📊 Session 3 Recap: Basic Tracking

In **Session 3**, we learned the fundamentals:

```python
# Basic tracking pattern from Session 3
with mlflow.start_run():
    # 1. Log parameters
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 10)
    
    # 2. Train model
    model = RandomForestClassifier(n_estimators=100, max_depth=10)
    model.fit(X_train, y_train)
    
    # 3. Log metrics
    accuracy = model.score(X_test, y_test)
    mlflow.log_metric("accuracy", accuracy)
    
    # 4. Log model
    mlflow.sklearn.log_model(model, "model")
```

**This works great for simple cases!**

---

### 🚀 Session 5: Advanced Patterns

Today we'll learn:

1. **Bulk parameter logging** - log 20+ params at once
2. **Autolog** - automatic logging for frameworks
3. **Complex artifacts** - plots, datasets, pipelines
4. **Nested runs** - parent-child hierarchies
5. **Advanced UI** - filtering, comparison, parallel coordinates
6. **Production patterns** - team workflows, reproducibility

**Let's see these in action!**

In [ ]:
# Cell 8: Advanced parameter logging demonstration

print("📝 Demonstrating Advanced Parameter Logging\n")

# Define a comprehensive configuration
model_config = {
    # Model hyperparameters
    'n_estimators': 100,
    'max_depth': 10,
    'min_samples_split': 5,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'bootstrap': True,
    
    # Training settings
    'random_state': 42,
    'test_size': 0.2,
    
    # Feature engineering
    'scaling_method': 'standard',
    'handle_imbalance': False,
    
    # Data info
    'dataset_version': 'v1.0',
    'n_features': X_train.shape[1]
}

print("🔹 OLD WAY (Session 3): 12 separate log_param() calls")
print("   mlflow.log_param('n_estimators', 100)")
print("   mlflow.log_param('max_depth', 10)")
print("   ... (10 more lines) ...")
print("   ❌ Repetitive, error-prone, hard to maintain\n")

print("🔹 NEW WAY (Session 5): 1 function call")
print("   log_params_dict(model_config)")
print("   ✅ Clean, maintainable, easy to version control\n")

print("💡 Benefits of bulk logging:")
print("   • Configs stored in code (version controlled)")
print("   • Easy to swap configurations")
print("   • Less code duplication")
print("   • Consistent naming conventions")

In [ ]:
# Cell 9: Complete tracking example with advanced patterns

print("🚀 Running Complete Advanced Tracking Example\n")

with mlflow.start_run(run_name="advanced_tracking_demo") as run:
    
    # 1. Log parameters using bulk method
    print("📝 Logging parameters...")
    log_params_dict(model_config)
    
    # 2. Train model using starter kit
    print("🤖 Training Random Forest...")
    model, metrics = train_random_forest(
        X_train, y_train, X_test, y_test,
        n_estimators=model_config['n_estimators'],
        max_depth=model_config['max_depth'],
        random_state=model_config['random_state'],
        log_to_mlflow=False  # We'll log manually for this demo
    )
    
    # 3. Log metrics using bulk method
    print("📊 Logging metrics...")
    log_metrics_dict(metrics)
    
    # 4. Generate and log visualizations
    print("📈 Generating visualizations...")
    
    # Get predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Log confusion matrix
    fig_cm = plot_confusion_matrix(y_test, y_pred)
    mlflow.log_figure(fig_cm, "confusion_matrix.png")
    plt.close(fig_cm)
    
    # Log ROC curve
    fig_roc = plot_roc_curve(y_test, y_pred_proba)
    mlflow.log_figure(fig_roc, "roc_curve.png")
    plt.close(fig_roc)
    
    # Log feature importance
    features = X_train.shape[1]
    fig_fi = plot_feature_importance(features, X_train.columns)
    mlflow.log_figure(fig_fi, "feature_importance.png")
    plt.close(fig_fi)
    
    # 5. Log model
    print("💾 Logging model...")
    mlflow.sklearn.log_model(model, "model")
    
    # 6. Set organizational tags
    print("🏷️  Setting tags...")
    set_tags_from_dict({
        'model_type': 'RandomForest',
        'framework': 'sklearn',
        'purpose': 'demo',
        'session': '5'
    })
    
    run_id = run.info.run_id
    
print(f"\n✅ Run completed successfully!")
# print(f"   Metrics: {metrics}")
# print(f"   Run ID: {run_id}")
# print(f"   Accuracy: {metrics['test_accuracy']:.4f}")
# print(f"   F1 Score: {metrics['test_f1']:.4f}")
# print(f"   ROC AUC: {metrics['test_roc_auc']:.4f}")

### 🎯 Try This in MLflow UI

Go to the MLflow UI and find the run "advanced_tracking_demo":

**Steps:**
1. Click on "Session5_Advanced_Tracking" experiment
2. Find the "advanced_tracking_demo" run
3. Explore the different tabs:
   - **Parameters**: See all 12 parameters logged at once
   - **Metrics**: View accuracy, F1, ROC AUC
   - **Artifacts**: Click to view plots (confusion matrix, ROC curve, feature importance)
   - **Tags**: See organizational metadata

**💡 Pro Tip**: Click on any plot artifact to view it full-screen!

---

**Questions to consider:**
- How is this different from Session 3's simple tracking?
- What additional information can you now track?
- How would this help in a team setting?

---

# Part 2: Autolog vs Manual Logging

⏱️ **Time**: 25 minutes

## The Power of Automatic Logging

One of MLflow's most powerful features is **autolog** - automatic logging for popular ML frameworks. But when should you use it vs manual logging?

### 🤔 The Real-World Problem

**Scenario**: Your team has 5 data scientists training hundreds of models per week.

**Without Autolog:**
- Everyone logs different things
- Inconsistent parameter names ("lr" vs "learning_rate")
- Some forget to log training time
- Hard to compare models

**With Autolog:**
- Consistent logging across team
- Comprehensive parameter capture
- Training metadata included
- Less code to maintain

### 📋 What Autolog Captures

Depending on the framework, autolog can capture:

✅ **Parameters**: All hyperparameters automatically  
✅ **Metrics**: Training/validation metrics  
✅ **Models**: The trained model artifact  
✅ **Metadata**: Training time, dataset info  
✅ **Model signature**: Input/output schema  
✅ **Requirements**: Package dependencies  

### 🎯 Supported Frameworks

- **scikit-learn**: `mlflow.sklearn.autolog()`
- **XGBoost**: `mlflow.xgboost.autolog()`
- **LightGBM**: `mlflow.lightgbm.autolog()`
- **Keras/TensorFlow**: `mlflow.tensorflow.autolog()`
- **PyTorch**: `mlflow.pytorch.autolog()`
- **And more!**

In [ ]:
# Cell 12: Autolog demonstration with sklearn

print("🔮 Demonstrating sklearn Autolog\n")

# Enable autolog for sklearn
mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True
)

print("✅ Autolog enabled for sklearn\n")
print("💡 Watch what happens - NO manual logging code needed!\n")

# Train model with ZERO manual logging
with mlflow.start_run(run_name="autolog_sklearn_demo") as run:
    
    from sklearn.ensemble import RandomForestClassifier
    
    print("🤖 Training RandomForestClassifier...")
    
    # Just train the model - autolog handles everything!
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # Evaluate (autolog captures this too!)
    score = model.score(X_test, y_test)
    
    run_id = run.info.run_id

print(f"\n✅ Training complete!")
print(f"   Run ID: {run_id}")
print(f"   Test Accuracy: {score:.4f}")
print("\n🎯 NOW: Check MLflow UI to see what was logged automatically!")
print("   Autolog captured: parameters, metrics, model, signature, and more!")

# Disable autolog to prevent interference with other examples
mlflow.sklearn.autolog(disable=True)
print("\n⚠️  Autolog disabled for remaining examples")

### 🎯 Try This in MLflow UI

**Exercise**: Compare the autolog run to the manual logging run

**Steps:**
1. Open MLflow UI
2. Find both runs:
   - "advanced_tracking_demo" (manual logging)
   - "autolog_sklearn_demo" (autolog)
3. Click on "autolog_sklearn_demo"
4. Explore what was captured:
   - **Parameters tab**: Are all hyperparameters there?
   - **Metrics tab**: What metrics were logged?
   - **Artifacts tab**: What files are present?
   - Look for: model signature, requirements.txt, input example

**Questions:**
- What did autolog capture that we logged manually?
- What did autolog capture that we DIDN'T log manually?
- Did you notice the model signature and input example?


In [ ]:
# Cell 14: Side-by-side comparison of manual vs autolog

print("📊 Side-by-Side Comparison: Manual vs Autolog\n")
print("="*70)

# Create comparison table
comparison_data = {
    'Aspect': [
        'Lines of Logging Code',
        'Parameters Logged',
        'Metrics Logged',
        'Model Logged',
        'Model Signature',
        'Input Example',
        'Requirements.txt',
        'Control/Customization',
        'Team Consistency',
        'Setup Time'
    ],
    'Manual Logging': [
        '~20 lines',
        'Only what you specify',
        'Only what you compute',
        'Yes (if you log it)',
        'No (unless manual)',
        'No (unless manual)',
        'No (unless manual)',
        '✅ Full control',
        '⚠️  Varies by person',
        '⏱️  Moderate'
    ],
    'Autolog': [
        '~1 line',
        'All hyperparameters',
        'Framework defaults',
        'Yes (automatic)',
        'Yes (automatic)',
        'Yes (automatic)',
        'Yes (automatic)',
        '⚠️  Less control',
        '✅ Always consistent',
        '⏱️  Fast'
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))
print("="*70)

print("\n🔑 Key Takeaways:\n")
print("✅ Autolog Advantages:")
print("   • Less code to write and maintain")
print("   • Consistent logging across team")
print("   • Comprehensive parameter capture")
print("   • Automatic model signatures")

print("\n⚠️  Autolog Limitations:")
print("   • Less flexibility for custom metrics")
print("   • May log more than you need")
print("   • Framework-specific behavior")
print("   • Can't customize artifact names easily")

print("\n💡 Best Practice: Use BOTH!")
print("   • Enable autolog for comprehensive baseline")
print("   • Add manual logging for custom metrics/artifacts")
print("   • Example: autolog + custom business metrics")

In [ ]:
# Cell 15: Decision framework - when to use each approach

print("🧭 Decision Framework: When to Use What\n")
print("="*70)

print("\n✅ USE AUTOLOG WHEN:\n")
scenarios_autolog = [
    ("Rapid experimentation", "Testing many models quickly"),
    ("Team consistency", "Multiple people training same model types"),
    ("Standard frameworks", "Using sklearn, XGBoost, TensorFlow, etc."),
    ("Learning/prototyping", "Focus on modeling, not tracking code")
]

for scenario, explanation in scenarios_autolog:
    print(f"   📌 {scenario}")
    print(f"      → {explanation}\n")

print("="*70)
print("\n⚙️  USE MANUAL LOGGING WHEN:\n")
scenarios_manual = [
    ("Custom metrics", "Business-specific KPIs (revenue impact, cost savings)"),
    ("Complex artifacts", "Domain-specific visualizations or reports"),
    ("Fine-grained control", "Specific naming conventions required"),
    ("Custom frameworks", "Using in-house or unsupported ML libraries")
]

for scenario, explanation in scenarios_manual:
    print(f"   📌 {scenario}")
    print(f"      → {explanation}\n")

print("="*70)
print("\n🎯 BEST PRACTICE: Hybrid Approach\n")

print("Example combining both:\n")
print("```python")
print("# Enable autolog for baseline logging")
print("mlflow.sklearn.autolog()")
print("")
print("with mlflow.start_run():")
print("    # Autolog handles standard sklearn logging")
print("    model.fit(X_train, y_train)")
print("    ")
print("    # Add custom business metrics manually")
print("    revenue_impact = calculate_revenue_impact(predictions)")
print("    mlflow.log_metric('revenue_impact_usd', revenue_impact)")
print("    ")
print("    # Add custom artifacts manually")
print("    business_report = generate_business_report(model, data)")
print("    mlflow.log_artifact('reports/business_analysis.pdf')")
print("```")

print("\n✅ Result: Comprehensive logging with custom insights!")

---

## 🔧 Autolog with Different Frameworks

Let's see how autolog works with **XGBoost** and **Keras/TensorFlow** to understand framework-specific differences.

In [ ]:
# Cell 24: Autolog with XGBoost

print("🚀 Autolog with XGBoost\n")

# Enable XGBoost autolog
mlflow.xgboost.autolog(
    log_input_examples=True,
    log_model_signatures=True
)

print("✅ XGBoost autolog enabled\n")

with mlflow.start_run(run_name="autolog_xgboost_demo") as run:
    
    import xgboost as xgb
    from sklearn.metrics import accuracy_score
    
    print("🤖 Training XGBoost Classifier...")
    
    # Prepare data in XGBoost format
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)
    
    # Set parameters
    params = {
        'max_depth': 6,
        'learning_rate': 0.1,
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'seed': 42
    }
    
    # Train with evaluation set (autolog captures training progress!)
    evals = [(dtrain, 'train'), (dtest, 'test')]
    model = xgb.train(
        params,
        dtrain,
        num_boost_round=100,
        evals=evals,
        early_stopping_rounds=10,
        verbose_eval=False
    )
    
    # Make predictions
    y_pred_proba = model.predict(dtest)
    y_pred = (y_pred_proba > 0.5).astype(int)
    accuracy = accuracy_score(y_test, y_pred)
    
    run_id = run.info.run_id

print(f"\n✅ XGBoost training complete!")
print(f"   Run ID: {run_id}")
print(f"   Test Accuracy: {accuracy:.4f}")

print("\n🔍 What XGBoost Autolog Captured:")
print("   ✅ All hyperparameters (max_depth, learning_rate, etc.)")
print("   ✅ Training metrics at each boosting round")
print("   ✅ Validation metrics (logloss for train & test)")
print("   ✅ Best iteration (from early stopping)")
print("   ✅ Model artifact with booster")
print("   ✅ Feature importance")

print("\n💡 XGBoost Specific Advantage:")
print("   → Captures metrics at EACH boosting round!")
print("   → Can visualize training curves directly in MLflow UI")
print("   → See exactly when early stopping triggered")

# Disable autolog
mlflow.xgboost.autolog(disable=True)

In [ ]:
# Cell 25: Autolog with Keras/TensorFlow

print("🧠 Autolog with Keras/TensorFlow\n")

# Enable TensorFlow autolog
mlflow.tensorflow.autolog(
    log_models=True,
    log_input_examples=True,
    # checkpoint=False  # Don't save checkpoints at each epoch
)

print("✅ TensorFlow autolog enabled\n")

with mlflow.start_run(run_name="autolog_keras_demo") as run:
    
    from tensorflow import keras
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Dropout
    from sklearn.preprocessing import StandardScaler
    
    print("🤖 Training Keras Neural Network...")
    
    # Scale features for neural network
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Build model
    model = Sequential([
        Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', 'AUC']
    )
    
    # Train (autolog captures training history!)
    history = model.fit(
        X_train_scaled, y_train,
        epochs=50,
        batch_size=32,
        validation_split=0.2,
        verbose=0
    )
    
    # Evaluate
    test_loss, test_acc, test_auc = model.evaluate(X_test_scaled, y_test, verbose=0)
    
    run_id = run.info.run_id

print(f"\n✅ Keras training complete!")
print(f"   Run ID: {run_id}")
print(f"   Test Accuracy: {test_acc:.4f}")
print(f"   Test AUC: {test_auc:.4f}")

print("\n🔍 What Keras Autolog Captured:")
print("   ✅ Model architecture summary")
print("   ✅ Optimizer configuration")
print("   ✅ Training metrics at each epoch")
print("   ✅ Validation metrics at each epoch")
print("   ✅ Learning curves (loss, accuracy over time)")
print("   ✅ Model in Keras format")
print("   ✅ TensorBoard logs")

print("\n💡 Keras Specific Advantages:")
print("   → Complete training history logged automatically")
print("   → Can create learning curves from UI")
print("   → Model architecture saved for reproducibility")
print("   → TensorBoard integration available")

# Disable autolog
mlflow.tensorflow.autolog(disable=True)

### 🎯 Try This in MLflow UI

**Exercise: Compare Training Behavior Across Frameworks**

You now have three autolog runs:
1. `autolog_sklearn_demo`
2. `autolog_xgboost_demo`
3. `autolog_keras_demo`

**In MLflow UI:**
1. Open each run and explore the **Metrics** tab
2. Notice the differences:
   - **sklearn**: Single point metrics (accuracy, etc.)
   - **XGBoost**: Metrics at each boosting round (see the curve!)
   - **Keras**: Metrics at each epoch (training & validation curves)

3. Click on metric names in XGBoost and Keras runs to see **training curves**
4. Look at the **Artifacts** tab in each:
   - Different model formats
   - Different metadata files

**Key Insight**: Autolog adapts to each framework's training paradigm!

---

💡 **Instructor Note**: Emphasize that iterative learners (XGBoost, Keras) get richer metric tracking automatically.

In [ ]:
# Cell 27: Common mistakes with autolog

print("⚠️  Common Mistakes with Autolog (and How to Avoid Them)\n")
print("="*70)

print("\n❌ MISTAKE #1: Forgetting to Disable Autolog\n")
print("Problem:")
print("   Autolog stays active for ALL subsequent runs")
print("   Can cause unexpected logging behavior")
print("")
print("Solution:")
print("   mlflow.sklearn.autolog(disable=True)  # Explicitly disable")
print("   Or: Use context manager approach (not common)")

print("\n" + "="*70)
print("\n❌ MISTAKE #2: Conflicts Between Autolog and Manual Logging\n")
print("Problem:")
print("   # This will log 'accuracy' TWICE")
print("   mlflow.sklearn.autolog()")
print("   with mlflow.start_run():")
print("       model.fit(X, y)  # Autolog logs accuracy")
print("       mlflow.log_metric('accuracy', score)  # You log it too!")
print("")
print("Solution:")
print("   • Use autolog for standard metrics")
print("   • Manually log only CUSTOM metrics")
print("   • Or disable specific autolog features:")
print("     mlflow.sklearn.autolog(log_models=False)  # Log params/metrics only")

print("\n" + "="*70)
print("\n❌ MISTAKE #3: Framework-Specific Quirks\n")
print("Problem:")
print("   Different frameworks have different autolog behaviors")
print("   Example: sklearn logs at fit(), XGBoost logs at train()")
print("")
print("Solution:")
print("   • Read docs for your specific framework")
print("   • Test with a simple example first")
print("   • Check MLflow UI to verify expected behavior")

print("\n" + "="*70)
print("\n❌ MISTAKE #4: Version Compatibility Issues\n")
print("Problem:")
print("   Autolog features vary by MLflow version")
print("   Some features require specific framework versions")
print("")
print("Solution:")
print("   • Keep MLflow updated: pip install --upgrade mlflow")
print("   • Check compatibility matrix in MLflow docs")
print("   • Pin versions in requirements.txt for reproducibility")

print("\n" + "="*70)
print("\n❌ MISTAKE #5: Assuming Autolog Logs Everything\n")
print("Problem:")
print("   Autolog captures framework-standard items only")
print("   Custom metrics, business KPIs, domain artifacts NOT logged")
print("")
print("Solution:")
print("   • Always combine autolog with manual logging for custom needs")
print("   • Example: autolog for model + manual for business impact")

print("\n" + "="*70)
print("\n✅ BEST PRACTICES:\n")
print("   1. Enable autolog at the start of your script")
print("   2. Disable explicitly when done")
print("   3. Add manual logging for custom needs")
print("   4. Verify in MLflow UI what was logged")
print("   5. Document your logging strategy for the team")

In [ ]:
# Cell 28: Hands-on exercise - Autolog + Custom Metrics

print("🎯 HANDS-ON EXERCISE: Combining Autolog with Custom Metrics\n")
print("="*70)

print("\n📋 TASK:")
print("   Train a LightGBM model with autolog enabled, then add custom business metrics")
print("")
print("🎯 REQUIREMENTS:")
print("   1. Enable LightGBM autolog")
print("   2. Train a LightGBM classifier")
print("   3. Let autolog handle standard logging")
print("   4. Manually log these custom metrics:")
print("      - false_positive_rate")
print("      - false_negative_rate")
print("      - cost_of_false_positives (assume $100 per FP)")
print("      - cost_of_false_negatives (assume $500 per FN)")
print("   5. Add a tag: 'exercise' = 'autolog_custom'")
print("   6. Verify everything in MLflow UI")

print("\n⏱️  Time: 5 minutes")
print("\n💡 Hint: Use starter kit's train_lightgbm_classifier() with log_to_mlflow=False")
print("   Then calculate and log custom metrics manually")

print("\n" + "="*70)
print("\n📝 YOUR CODE BELOW (try it yourself first!)")
print("\n" + "="*70)

# Students try first, then see solution below

print("\n\n✅ SOLUTION:\n")

# Enable LightGBM autolog
mlflow.lightgbm.autolog(log_input_examples=True, log_model_signatures=True)

with mlflow.start_run(run_name="exercise_autolog_custom") as run:
    
    import lightgbm as lgb
    from sklearn.metrics import confusion_matrix
    
    print("🤖 Training LightGBM (autolog handles standard logging)...")
    
    # Prepare data
    train_data = lgb.Dataset(X_train, label=y_train)
    test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)
    
    # Parameters
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'seed': 42
    }
    
    # Train (autolog captures everything!)
    model = lgb.train(
        params,
        train_data,
        num_boost_round=100,
        valid_sets=[test_data],
        valid_names=['test']
    )
    
    # Make predictions
    y_pred_proba = model.predict(X_test)
    y_pred = (y_pred_proba > 0.5).astype(int)
    
    print("\n📊 Computing custom business metrics...")
    
    # Calculate confusion matrix
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    
    # Custom metrics
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    cost_fp = fp * 100  # $100 per false positive
    cost_fn = fn * 500  # $500 per false negative
    total_cost = cost_fp + cost_fn
    
    # Manually log custom metrics
    mlflow.log_metrics({
        'false_positive_rate': fpr,
        'false_negative_rate': fnr,
        'cost_false_positives_usd': cost_fp,
        'cost_false_negatives_usd': cost_fn,
        'total_cost_usd': total_cost
    })
    
    # Add tag
    mlflow.set_tag('exercise', 'autolog_custom')
    
    run_id = run.info.run_id

print(f"\n✅ Exercise Complete!")
print(f"   Run ID: {run_id}")
print(f"   False Positive Rate: {fpr:.4f}")
print(f"   False Negative Rate: {fnr:.4f}")
print(f"   Total Cost: ${total_cost:,.2f}")

print("\n🎯 NOW: Verify in MLflow UI")
print("   1. Find run 'exercise_autolog_custom'")
print("   2. Check standard LightGBM metrics (from autolog)")
print("   3. Check custom business metrics (your manual logging)")
print("   4. Verify the 'exercise' tag")

# Disable autolog
mlflow.lightgbm.autolog(disable=True)

print("\n💡 Key Insight: Autolog + Manual = Complete Tracking!")

---

### ✅ Part 2 Complete: Autolog Mastery

**What you learned:**
- ✅ How autolog works with sklearn, XGBoost, LightGBM, and Keras
- ✅ When to use autolog vs manual logging
- ✅ How to combine both for comprehensive tracking
- ✅ Common mistakes and how to avoid them
- ✅ Practical exercise with business metrics

**Next up**: Advanced artifacts and the critical skill of dataset versioning! 🚀

---

# Part 3: Advanced Artifacts & Dataset Versioning

⏱️ **Time**: 30 minutes

## 🎯 Why Artifacts Matter in Production

### The Real-World Story

**Scenario**: Your customer churn model is in production for 6 months.

**Week 1**: Accuracy = 0.92, everyone's happy ✅  
**Week 24**: Accuracy = 0.78, stakeholders are concerned ⚠️

**The Question**: *"What changed?"*

**Without Proper Artifacts:**
- ❌ "Which dataset did we use?"
- ❌ "What preprocessing steps were applied?"
- ❌ "Which features were included?"
- ❌ "Can we reproduce the original model?"

**You're stuck. Team scrambles for 2 weeks. Production impact: $50K.**

---

**With Comprehensive Artifacts:**
- ✅ Dataset version: v1.0 (with hash)
- ✅ Preprocessing pipeline: saved as artifact
- ✅ Feature list: tracked in metadata
- ✅ Training plots: confusion matrix shows class imbalance change
- ✅ Data quality report: logs reveal data drift

**Root cause identified in 2 hours. Fixed in 1 day. Crisis averted.** ✅

---

### 📦 Types of Artifacts

**Standard Artifacts:**
- Models (pkl, h5, pb)
- Plots (confusion matrix, ROC curves)
- Metrics reports

**Advanced Artifacts** (Today's Focus):
- **Datasets**: Versioned data snapshots
- **Pipelines**: Preprocessing transformations
- **Reports**: Data quality, feature analysis
- **Configurations**: JSON, YAML files
- **Documentation**: Model cards, analysis docs

### 📊 Visualization Artifacts Recap

We've already seen how to log plots (Part 1). Let's quickly review:

In [ ]:
# Cell 30: Visualization artifacts review

print("📈 Quick Review: Logging Visualization Artifacts\n")

with mlflow.start_run(run_name="viz_artifacts_demo") as run:
    
    # Train a quick model
    print("🤖 Training model...")
    model, metrics = train_random_forest(
        X_train, y_train, X_test, y_test,
        n_estimators=100,
        max_depth=10,
        log_to_mlflow=False
    )
    
    # Get predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    print("\n📊 Generating and logging visualizations...")
    
    # 1. Confusion Matrix
    fig_cm = plot_confusion_matrix(y_test, y_pred)
    mlflow.log_figure(fig_cm, "confusion_matrix.png")
    print("   ✅ Confusion matrix logged")
    plt.close(fig_cm)
    
    # 2. ROC Curve
    fig_roc = plot_roc_curve(y_test, y_pred_proba)
    mlflow.log_figure(fig_roc, "roc_curve.png")
    print("   ✅ ROC curve logged")
    plt.close(fig_roc)
    
    # 3. Feature Importance
    features = X_train.shape[1]
    fig_fi = plot_feature_importance(features, X_train.columns, top_n=15)
    mlflow.log_figure(fig_fi, "feature_importance.png")
    print("   ✅ Feature importance logged")
    plt.close(fig_fi)
    
    # 4. Precision-Recall Curve
    fig_pr = plot_precision_recall_curve(y_test, y_pred_proba)
    mlflow.log_figure(fig_pr, "precision_recall_curve.png")
    print("   ✅ Precision-recall curve logged")
    plt.close(fig_pr)
    
    run_id = run.info.run_id

print(f"\n✅ All visualizations logged!")
print(f"   Run ID: {run_id}")
print("\n💡 Go to MLflow UI → Artifacts tab to view all plots")

---

## 🔑 CRITICAL SKILL: Dataset Versioning

This is one of the **most important practices** for production ML systems. Let's master it.

### 🎯 Why Dataset Versioning is Critical

**Real Production Story:**

A fraud detection model shows declining performance after 3 months in production.

**Investigation reveals:**
- Training data: January dataset (100K transactions, 5% fraud rate)
- Production data: April dataset (150K transactions, 8% fraud rate)
- New fraud patterns emerged
- Feature distributions changed

**Without dataset versioning:** It took 2 weeks to figure this out
**With dataset versioning:** Would have been obvious in 30 minutes

---

### 📋 What to Track About Datasets

**Essential Metadata:**
1. **Version ID**: v1.0, v2.1, 2024-01-15
2. **Data Hash**: MD5 or SHA256 checksum
3. **Size**: Row count, column count, file size
4. **Collection Date**: When data was collected
5. **Source**: Where data came from
6. **Features**: List of feature names
7. **Target Distribution**: Class balance
8. **Data Quality**: Missing values, outliers

**Why each matters:**
- **Hash**: Guarantees exact same data
- **Collection date**: Explains temporal patterns
- **Target distribution**: Detects class imbalance changes
- **Feature list**: Tracks feature engineering evolution

In [ ]:
# Cell 31: Dataset versioning - metadata tracking

print("🗂️  Dataset Versioning: Metadata Tracking\n")
print("="*70)

import hashlib
import json
from datetime import datetime
import os

with mlflow.start_run(run_name="dataset_versioning_demo") as run:
    
    print("📊 Calculating dataset metadata...\n")
    
    # 1. Calculate dataset hash (checksum)
    dataset_path = 'data/customer_churn.csv'
    
    with open(dataset_path, 'rb') as f:
        file_bytes = f.read()
        dataset_hash = hashlib.md5(file_bytes).hexdigest()
    
    print(f"🔐 Dataset Hash (MD5): {dataset_hash[:16]}...")
    
    # 2. Gather comprehensive metadata
    dataset_metadata = {
        # Version info
        'dataset_version': 'v1.0',
        'dataset_hash': dataset_hash,
        'collection_date': '2024-01-15',
        
        # Size info
        'total_rows': len(X_train) + len(X_test),
        'train_rows': len(X_train),
        'test_rows': len(X_test),
        'n_features': X_train.shape[1],
        'file_size_mb': round(os.path.getsize(dataset_path) / (1024*1024), 2),
        
        # Feature info
        'feature_names': X_train.columns.tolist(),
        
        # Target distribution
        'train_churn_rate': float(y_train.mean()),
        'test_churn_rate': float(y_test.mean()),
        'train_churn_count': int(y_train.sum()),
        'train_no_churn_count': int((y_train == 0).sum()),
        
        # Data quality
        'train_missing_values': int(X_train.isnull().sum().sum()),
        'test_missing_values': int(X_test.isnull().sum().sum()),
        
        # Source info
        'data_source': 'CRM Database',
        'dataset_name': 'customer_churn',
        'processed_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    print("\n📋 Dataset Metadata:")
    for key, value in list(dataset_metadata.items())[:10]:  # Show first 10
        if key != 'feature_names':  # Skip feature list for brevity
            print(f"   {key}: {value}")
    print(f"   ... and {len(dataset_metadata) - 10} more fields")
    
    # 3. Log metadata as parameters
    # Note: MLflow params have size limits, so be selective
    params_to_log = {
        'dataset_version': dataset_metadata['dataset_version'],
        'dataset_hash': dataset_metadata['dataset_hash'][:32],  # Truncated
        'collection_date': dataset_metadata['collection_date'],
        'total_rows': dataset_metadata['total_rows'],
        'n_features': dataset_metadata['n_features'],
        'train_churn_rate': round(dataset_metadata['train_churn_rate'], 4),
        'data_source': dataset_metadata['data_source']
    }
    
    log_params_dict(params_to_log)
    print("\n✅ Key metadata logged as parameters")
    
    # 4. Save complete metadata as JSON artifact
    metadata_file = 'dataset_metadata.json'
    with open(metadata_file, 'w') as f:
        json.dump(dataset_metadata, f, indent=2)
    
    mlflow.log_artifact(metadata_file, artifact_path='metadata')
    os.remove(metadata_file)  # Clean up local file
    print("✅ Complete metadata saved as JSON artifact")
    
    # Train a model so we have complete run
    model, metrics = train_random_forest(
        X_train, y_train, X_test, y_test,
        n_estimators=100,
        log_to_mlflow=False
    )
    log_metrics_dict(metrics)
    
    run_id = run.info.run_id

print(f"\n✅ Dataset versioning complete!")
print(f"   Run ID: {run_id}")

print("\n🔍 What We Achieved:")
print("   ✅ Unique hash identifies exact dataset")
print("   ✅ Comprehensive metadata tracked")
print("   ✅ Key params searchable in MLflow")
print("   ✅ Full details in JSON artifact")

print("\n💡 Future Use Case:")
print("   6 months later: Model performance drops")
print("   → Check dataset_hash: Did data change?")
print("   → Check collection_date: Temporal shift?")
print("   → Check train_churn_rate: Class imbalance change?")
print("   → Root cause identified in minutes!")

In [ ]:
# Cell 32: Log dataset as artifact

print("💾 Logging Dataset as Artifact\n")
print("="*70)

with mlflow.start_run(run_name="dataset_artifact_demo") as run:
    
    print("📂 Saving dataset snapshot...\n")
    
    # Method 1: Log the original CSV file
    print("Method 1: Log original CSV")
    mlflow.log_artifact('data/customer_churn.csv', artifact_path='datasets')
    print("   ✅ Original dataset logged\n")
    
    # Method 2: Log processed train/test splits
    print("Method 2: Log train/test splits")
    
    # Save train data
    train_df = X_train.copy()
    train_df['target'] = y_train
    train_df.to_csv('train_data.csv', index=False)
    mlflow.log_artifact('train_data.csv', artifact_path='datasets/splits')
    print("   ✅ Training split logged")
    os.remove('train_data.csv')
    
    # Save test data
    test_df = X_test.copy()
    test_df['target'] = y_test
    test_df.to_csv('test_data.csv', index=False)
    mlflow.log_artifact('test_data.csv', artifact_path='datasets/splits')
    print("   ✅ Test split logged\n")
    os.remove('test_data.csv')
    
    # Method 3: Log dataset statistics summary
    print("Method 3: Log dataset summary statistics")
    
    summary = {
        'numerical_features': X_train.describe().to_dict(),
        'categorical_features': {
            col: X_train[col].value_counts().to_dict()
            for col in X_train.select_dtypes(include=['object']).columns
        },
        'target_distribution': {
            'train': y_train.value_counts().to_dict(),
            'test': y_test.value_counts().to_dict()
        },
        'missing_values': {
            'train': X_train.isnull().sum().to_dict(),
            'test': X_test.isnull().sum().to_dict()
        }
    }
    
    with open('dataset_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    mlflow.log_artifact('dataset_summary.json', artifact_path='datasets')
    print("   ✅ Summary statistics logged\n")
    os.remove('dataset_summary.json')
    
    # Log dataset version as parameter
    mlflow.log_param('dataset_version', 'v1.0')
    mlflow.log_param('dataset_logged', True)
    
    # Train model
    model, metrics = train_random_forest(
        X_train, y_train, X_test, y_test,
        n_estimators=100,
        log_to_mlflow=False
    )
    log_metrics_dict(metrics)
    mlflow.sklearn.log_model(model, "model")
    
    run_id = run.info.run_id

print(f"\n✅ Dataset artifacts logged!")
print(f"   Run ID: {run_id}")

print("\n📦 What's in MLflow:")
print("   datasets/")
print("   ├── customer_churn.csv (original data)")
print("   ├── dataset_summary.json (statistics)")
print("   └── splits/")
print("       ├── train_data.csv")
print("       └── test_data.csv")

print("\n💡 Why This Matters:")
print("   • Exact data snapshot preserved")
print("   • Can reproduce results months later")
print("   • Debug data issues easily")
print("   • Audit compliance (know exactly what data was used)")

print("\n🎯 TRY NOW in MLflow UI:")
print("   1. Find 'dataset_artifact_demo' run")
print("   2. Go to Artifacts tab")
print("   3. Navigate to datasets/ folder")
print("   4. Download train_data.csv")
print("   5. Open dataset_summary.json")

In [ ]:
# Cell 33: Data lineage tracking

print("🔄 Data Lineage Tracking\n")
print("="*70)

print("\n💡 What is Data Lineage?")
print("   → Complete history of your data from source to model")
print("   → 'Data genealogy' - who, what, when, where, how\n")

with mlflow.start_run(run_name="data_lineage_demo") as run:
    
    print("📝 Tracking data lineage...\n")
    
    # 1. Track data source
    data_lineage = {
        # Origin
        'source_system': 'CRM Database',
        'source_table': 'customers',
        'extraction_date': '2025-10-15',
        'extracted_by': 'data_pipeline_v2',
        
        # Transformations applied
        'transformations': [
            '1. Removed duplicate customer IDs',
            '2. Handled missing values (mean imputation)',
            '3. Encoded categorical variables (one-hot)',
            '4. Scaled numerical features (StandardScaler)',
            '5. Split 80/20 train/test',
        ],
        
        # Data quality checks
        'quality_checks': {
            'null_check': 'passed',
            'duplicate_check': 'passed',
            'outlier_check': 'passed',
            'schema_validation': 'passed'
        },
        
        # Preprocessing code version (from Session 4 - Git!)
        'preprocessing_script': 'preprocessing.py',
        'git_commit': 'abc123def456',  # Link to Session 4!
        'preprocessing_version': 'v2.1',
        
        # Dependencies
        'feature_dependencies': {
            'tenure': ['contract_start_date', 'current_date'],
            'avg_monthly_spend': ['total_charges', 'tenure'],
            'payment_method_encoded': ['payment_method']
        }
    }
    
    print("🔍 Lineage Information:")
    print(f"   Source: {data_lineage['source_system']}")
    print(f"   Extracted: {data_lineage['extraction_date']}")
    print(f"   Transformations: {len(data_lineage['transformations'])} steps")
    print(f"   Quality checks: {len(data_lineage['quality_checks'])} passed")
    print(f"   Code version: {data_lineage['preprocessing_version']}")
    
    # Log as parameters (key info only)
    mlflow.log_params({
        'data_source': data_lineage['source_system'],
        'extraction_date': data_lineage['extraction_date'],
        'preprocessing_version': data_lineage['preprocessing_version'],
        'git_commit': data_lineage['git_commit']
    })
    
    # Log complete lineage as artifact
    with open('data_lineage.json', 'w') as f:
        json.dump(data_lineage, f, indent=2)
    
    mlflow.log_artifact('data_lineage.json', artifact_path='lineage')
    os.remove('data_lineage.json')
    
    print("\n✅ Data lineage tracked!")
    
    # Log transformation pipeline description
    pipeline_doc = """
# Data Preprocessing Pipeline v2.1

## Steps:
1. **Data Extraction**
   - Source: CRM Database (customers table)
   - Date: 2025-10-15
   - Rows extracted: 10,000

2. **Data Cleaning**
   - Removed 47 duplicate customer IDs
   - Imputed 123 missing values (mean strategy)
   - Validated schema: PASSED

3. **Feature Engineering**
   - Created 'tenure' from date diff
   - Calculated 'avg_monthly_spend'
   - One-hot encoded 3 categorical variables

4. **Feature Scaling**
   - StandardScaler applied to numerical features
   - Fit on train, transform on test

5. **Train/Test Split**
   - Split ratio: 80/20
   - Stratified by target
   - Random seed: 42
    """
    
    with open('preprocessing_pipeline.md', 'w') as f:
        f.write(pipeline_doc)
    
    mlflow.log_artifact('preprocessing_pipeline.md', artifact_path='lineage')
    os.remove('preprocessing_pipeline.md')
    
    # Train model
    model, metrics = train_random_forest(
        X_train, y_train, X_test, y_test,
        n_estimators=100,
        log_to_mlflow=False
    )
    log_metrics_dict(metrics)
    mlflow.sklearn.log_model(model, "model")
    
    run_id = run.info.run_id

print(f"\n✅ Complete lineage tracked!")
print(f"   Run ID: {run_id}")

print("\n🎯 Why Lineage Matters:")
print("")
print("   Scenario 1: Model performance degrades")
print("   → Check data_lineage.json: Did source data change?")
print("   → Check extraction_date: Using old data?")
print("")
print("   Scenario 2: Need to reproduce model")
print("   → Use git_commit to get exact preprocessing code")
print("   → Use preprocessing_pipeline.md for steps")
print("   → Use extraction_date to get same data period")
print("")
print("   Scenario 3: Audit compliance")
print("   → Complete audit trail from source to model")
print("   → Quality checks documented")
print("   → Transformations traceable")

print("\n💡 Connection to Session 4 (Git):")
print("   By logging git_commit, you link ML runs to code versions")
print("   Full reproducibility: code + data + environment!")

---

## 🔧 Advanced Artifact Patterns

Beyond datasets and plots, let's explore other critical artifacts for production ML.

In [ ]:
# Cell 34: Log preprocessing pipeline as artifact

print("⚙️  Logging Preprocessing Pipeline\n")
print("="*70)

with mlflow.start_run(run_name="pipeline_artifact_demo") as run:
    
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import Pipeline
    from sklearn.ensemble import RandomForestClassifier
    import joblib
    
    print("🔧 Creating preprocessing pipeline...\n")
    
    # Create a complete pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
    
    # Train the pipeline
    print("🤖 Training pipeline...")
    pipeline.fit(X_train, y_train)
    score = pipeline.score(X_test, y_test)
    print(f"   Accuracy: {score:.4f}\n")
    
    # Method 1: Log entire pipeline
    print("Method 1: Log complete pipeline")
    mlflow.sklearn.log_model(pipeline, "pipeline")
    print("   ✅ Full pipeline logged (scaler + model)\n")
    
    # Method 2: Log scaler separately (for reuse)
    print("Method 2: Log scaler separately")
    scaler_path = 'preprocessing_scaler.pkl'
    joblib.dump(pipeline.named_steps['scaler'], scaler_path)
    mlflow.log_artifact(scaler_path, artifact_path='preprocessing')
    print("   ✅ Scaler saved separately\n")
    os.remove(scaler_path)
    
    # Method 3: Log scaler parameters
    print("Method 3: Log scaler metadata")
    scaler_metadata = {
        'scaler_type': 'StandardScaler',
        'mean': pipeline.named_steps['scaler'].mean_.tolist(),
        'std': pipeline.named_steps['scaler'].scale_.tolist(),
        'n_features': len(pipeline.named_steps['scaler'].mean_),
        'feature_names': X_train.columns.tolist()
    }
    
    with open('scaler_metadata.json', 'w') as f:
        json.dump(scaler_metadata, f, indent=2)
    
    mlflow.log_artifact('scaler_metadata.json', artifact_path='preprocessing')
    print("   ✅ Scaler metadata logged\n")
    os.remove('scaler_metadata.json')
    
    # Log key parameters
    mlflow.log_params({
        'preprocessing_type': 'StandardScaler',
        'pipeline_steps': 'scaler->classifier'
    })
    
    mlflow.log_metrics({
        'test_accuracy': score
    })
    
    run_id = run.info.run_id

print(f"\n✅ Preprocessing pipeline logged!")
print(f"   Run ID: {run_id}")

print("\n💡 Why Log Preprocessing Pipelines?\n")
print("   Scenario 1: Deploy to production")
print("   → Load pipeline, apply same scaling to new data")
print("   → Prevents train/serve skew\n")

print("   Scenario 2: Reproduce results")
print("   → Exact same transformations applied")
print("   → No guessing about preprocessing steps\n")

print("   Scenario 3: Debug issues")
print("   → Check scaler.mean_ values")
print("   → Verify feature order")
print("   → Ensure consistent preprocessing")

print("\n🎯 TRY in MLflow UI:")
print("   1. Find 'pipeline_artifact_demo' run")
print("   2. Artifacts → preprocessing/")
print("   3. Download preprocessing_scaler.pkl")
print("   4. View scaler_metadata.json")
print("   5. See complete pipeline in 'pipeline' folder")

In [ ]:
# Cell 35: Log multiple artifacts and directories

print("📦 Logging Multiple Artifacts and Directories\n")
print("="*70)

import tempfile
import shutil

with mlflow.start_run(run_name="multi_artifact_demo") as run:
    
    print("📂 Creating artifact structure...\n")
    
    # Create a temporary directory structure
    temp_dir = tempfile.mkdtemp()
    
    # 1. Create reports directory
    reports_dir = os.path.join(temp_dir, 'reports')
    os.makedirs(reports_dir, exist_ok=True)
    
    # Model performance report
    with open(os.path.join(reports_dir, 'model_performance.txt'), 'w') as f:
        f.write("Model Performance Summary\n")
        f.write("=" * 40 + "\n")
        f.write("Model: Random Forest Classifier\n")
        f.write("Dataset: Customer Churn v1.0\n")
        f.write("Training Date: 2025-10-15\n")
        f.write("\nMetrics:\n")
        f.write("  Accuracy: 0.8542\n")
        f.write("  Precision: 0.8231\n")
        f.write("  Recall: 0.7891\n")
        f.write("  F1 Score: 0.8057\n")
    
    # Data quality report
    with open(os.path.join(reports_dir, 'data_quality.txt'), 'w', encoding='utf-8') as f:
        f.write("Data Quality Report\n")
        f.write("=" * 40 + "\n")
        f.write("Dataset: customer_churn.csv\n")
        f.write("Total Rows: 10,000\n")
        f.write("\nQuality Checks:\n")
        f.write("  ✓ No duplicate IDs\n")
        f.write("  ✓ Missing values: 0.02%\n")
        f.write("  ✓ Outliers detected: 3% (within threshold)\n")
        f.write("  ✓ Schema validation: PASSED\n")
    
    print("✅ Created reports/")
    
    # 2. Create configs directory
    configs_dir = os.path.join(temp_dir, 'configs')
    os.makedirs(configs_dir, exist_ok=True)
    
    # Model config
    model_config = {
        'model_type': 'RandomForestClassifier',
        'hyperparameters': {
            'n_estimators': 100,
            'max_depth': 10,
            'min_samples_split': 5,
            'random_state': 42
        },
        'training': {
            'test_size': 0.2,
            'stratify': True,
            'cross_validation': False
        }
    }
    
    with open(os.path.join(configs_dir, 'model_config.json'), 'w') as f:
        json.dump(model_config, f, indent=2)
    
    # Feature config
    feature_config = {
        'numerical_features': ['tenure', 'monthly_charges', 'total_charges'],
        'categorical_features': ['contract_type', 'payment_method', 'internet_service'],
        'feature_engineering': [
            'avg_monthly_charges = total_charges / tenure',
            'is_long_tenure = tenure > 24'
        ],
        'scaling': 'StandardScaler',
        'encoding': 'OneHotEncoder'
    }
    
    with open(os.path.join(configs_dir, 'feature_config.json'), 'w') as f:
        json.dump(feature_config, f, indent=2)
    
    print("✅ Created configs/")
    
    # 3. Create analysis directory
    analysis_dir = os.path.join(temp_dir, 'analysis')
    os.makedirs(analysis_dir, exist_ok=True)
    
    # Feature importance analysis
    with open(os.path.join(analysis_dir, 'feature_analysis.md'), 'w', encoding='utf-8') as f:
        f.write("# Feature Importance Analysis\n\n")
        f.write("## Top 5 Features\n\n")
        f.write("1. **tenure** (importance: 0.234)\n")
        f.write("   - Most predictive feature\n")
        f.write("   - Customers with tenure < 12 months 3x more likely to churn\n\n")
        f.write("2. **monthly_charges** (importance: 0.189)\n")
        f.write("   - Higher charges correlate with churn\n\n")
        f.write("3. **contract_type** (importance: 0.156)\n")
        f.write("   - Month-to-month contracts highest risk\n\n")
    
    print("✅ Created analysis/\n")
    
    # Log all directories
    print("📤 Uploading to MLflow...\n")
    
    mlflow.log_artifacts(reports_dir, artifact_path='reports')
    print("   ✅ Logged reports/ directory")
    
    mlflow.log_artifacts(configs_dir, artifact_path='configs')
    print("   ✅ Logged configs/ directory")
    
    mlflow.log_artifacts(analysis_dir, artifact_path='analysis')
    print("   ✅ Logged analysis/ directory")
    
    # Train model
    model, metrics = train_random_forest(
        X_train, y_train, X_test, y_test,
        n_estimators=100,
        log_to_mlflow=False
    )
    log_metrics_dict(metrics)
    mlflow.sklearn.log_model(model, "model")
    
    # Clean up temp directory
    shutil.rmtree(temp_dir)
    
    run_id = run.info.run_id

print(f"\n✅ Multiple artifacts logged!")
print(f"   Run ID: {run_id}")

print("\n📂 Artifact Structure in MLflow:")
print("")
print("   artifacts/")
print("   ├── reports/")
print("   │   ├── model_performance.txt")
print("   │   └── data_quality.txt")
print("   ├── configs/")
print("   │   ├── model_config.json")
print("   │   └── feature_config.json")
print("   ├── analysis/")
print("   │   └── feature_analysis.md")
print("   └── model/")
print("       └── (sklearn model files)")

print("\n💡 Professional Artifact Organization:")
print("   ✓ Clear directory structure")
print("   ✓ Reports for stakeholders")
print("   ✓ Configs for reproducibility")
print("   ✓ Analysis for insights")
print("   ✓ Everything version controlled with the run")

In [ ]:
# Cell 36: Custom artifact patterns - JSON configs and reports

print("📄 Custom Artifact Patterns\n")
print("="*70)

with mlflow.start_run(run_name="custom_artifacts_demo") as run:
    
    print("📝 Creating custom artifacts...\n")
    
    # 1. Experiment configuration
    experiment_config = {
        'experiment': {
            'name': 'customer_churn_prediction',
            'date': '2024-01-15',
            'researcher': 'ML Team',
            'hypothesis': 'Random Forest will outperform logistic regression'
        },
        'data': {
            'source': 'CRM Database',
            'version': 'v1.0',
            'samples': 10000,
            'features': 19
        },
        'model': {
            'algorithm': 'RandomForestClassifier',
            'framework': 'sklearn',
            'version': '1.3.0'
        },
        'compute': {
            'environment': 'local',
            'cpu_cores': 8,
            'memory_gb': 16,
            'gpu': False
        }
    }
    
    with open('experiment_config.json', 'w') as f:
        json.dump(experiment_config, f, indent=2)
    
    mlflow.log_artifact('experiment_config.json')
    os.remove('experiment_config.json')
    print("✅ Experiment config logged")
    
    # 2. Model card (documentation)
    model_card = """
# Model Card: Customer Churn Classifier

## Model Details
- **Model Type**: Random Forest Classifier
- **Framework**: scikit-learn 1.3.0
- **Training Date**: 2025-10-15
- **Version**: 1.0

## Intended Use
- **Primary Use**: Predict customer churn probability
- **Users**: Customer success team, marketing team
- **Out of Scope**: Not for individual customer decisions without human review

## Training Data
- **Dataset**: CRM customer records (v1.0)
- **Size**: 10,000 customers
- **Date Range**: 2024-01-01 to 2025-10-15
- **Class Distribution**: 26.5% churn, 73.5% retained

## Performance
- **Test Accuracy**: 85.4%
- **Precision**: 82.3%
- **Recall**: 78.9%
- **F1 Score**: 80.6%
- **ROC AUC**: 0.912

## Limitations
- Trained on historical data, may not capture new churn patterns
- Performance degrades for customers with tenure < 1 month
- Does not account for external factors (economic conditions, competitors)

## Ethical Considerations
- Model should not be sole factor in customer treatment decisions
- Regular monitoring for bias across customer segments required
- Privacy: Uses only authorized customer data
    """
    
    with open('MODEL_CARD.md', 'w') as f:
        f.write(model_card)
    
    mlflow.log_artifact('MODEL_CARD.md')
    os.remove('MODEL_CARD.md')
    print("✅ Model card logged")
    
    # 3. Data quality report
    data_quality_report = {
        'timestamp': '2025-15-15T10:30:00',
        'dataset': 'customer_churn_v1.0',
        'checks': {
            'completeness': {
                'status': 'PASS',
                'missing_values_pct': 0.02,
                'threshold': 5.0
            },
            'uniqueness': {
                'status': 'PASS',
                'duplicate_ids': 0,
                'threshold': 0
            },
            'validity': {
                'status': 'PASS',
                'invalid_values': 3,
                'threshold': 100
            },
            'consistency': {
                'status': 'PASS',
                'inconsistent_records': 0,
                'threshold': 50
            }
        },
        'statistics': {
            'total_rows': 10000,
            'total_columns': 20,
            'numerical_columns': 16,
            'categorical_columns': 4
        },
        'recommendations': [
            'Data quality is excellent - ready for production',
            'Monitor missing value rate weekly',
            'Set up alerts if duplicate IDs detected'
        ]
    }
    
    with open('data_quality_report.json', 'w') as f:
        json.dump(data_quality_report, f, indent=2)
    
    mlflow.log_artifact('data_quality_report.json', artifact_path='quality')
    os.remove('data_quality_report.json')
    print("✅ Data quality report logged")
    
    # Train model
    model, metrics = train_random_forest(
        X_train, y_train, X_test, y_test,
        n_estimators=100,
        log_to_mlflow=False
    )
    log_metrics_dict(metrics)
    mlflow.sklearn.log_model(model, "model")
    
    run_id = run.info.run_id

print(f"\n✅ Custom artifacts logged!")
print(f"   Run ID: {run_id}")

print("\n📄 Custom Artifacts Created:")
print("   • experiment_config.json - Complete experiment details")
print("   • MODEL_CARD.md - Model documentation")
print("   • data_quality_report.json - Data quality checks")

print("\n💡 Why These Matter:")
print("")
print("   📋 Experiment Config:")
print("      → Reproducibility: Know exact environment")
print("      → Collaboration: Team understands context")
print("")
print("   📝 Model Card:")
print("      → Documentation: Clear model purpose and limits")
print("      → Governance: Ethics and compliance")
print("      → Stakeholder communication")
print("")
print("   ✓ Quality Report:")
print("      → Trust: Data quality validated")
print("      → Monitoring: Track quality over time")
print("      → Debugging: Identify data issues fast")

---

## 🎯 Hands-On Exercise: Log 3 Different Artifacts

Time to practice what you've learned!

In [ ]:
# Cell 37: Hands-on exercise - Log 3 different artifacts

print("🎯 HANDS-ON EXERCISE: Comprehensive Artifact Logging\n")
print("="*70)

print("\n📋 TASK:")
print("   Train a model and log THREE different types of artifacts")
print("")
print("🎯 REQUIREMENTS:")
print("")
print("   1. ARTIFACT TYPE 1: Visualization")
print("      → Log confusion matrix AND ROC curve")
print("")
print("   2. ARTIFACT TYPE 2: Dataset Versioning")
print("      → Create dataset_metadata dict with:")
print("        - version, hash, rows, features, target_distribution")
print("      → Log metadata as JSON artifact")
print("      → Log key metadata as parameters")
print("")
print("   3. ARTIFACT TYPE 3: Preprocessing Pipeline")
print("      → Train a model with StandardScaler")
print("      → Save and log the scaler as pickle file")
print("      → Log scaler parameters as JSON")
print("")
print("   4. ORGANIZATION:")
print("      → Use proper artifact_path for each type")
print("      → visualizations/ for plots")
print("      → datasets/ for dataset info")
print("      → preprocessing/ for pipeline artifacts")
print("")
print("   5. VERIFICATION:")
print("      → Add tag: 'exercise' = 'artifact_logging'")
print("      → Verify all artifacts in MLflow UI")

print("\n⏱️  Time: 10 minutes")
print("\n💡 Hints:")
print("   • Use train_random_forest() with log_to_mlflow=False")
print("   • Calculate hash: hashlib.md5(open(file, 'rb').read()).hexdigest()")
print("   • Save scaler: joblib.dump(scaler, 'scaler.pkl')")
print("   • Use mlflow.log_figure() for plots")
print("   • Use mlflow.log_artifact(file, artifact_path='folder/')")

print("\n" + "="*70)
print("\n📝 YOUR CODE BELOW (try it yourself first!)")
print("\n" + "="*70)
print("\n\n")

In [ ]:
# Cell 38: Exercise solution

print("✅ SOLUTION: Comprehensive Artifact Logging\n")
print("="*70)

import hashlib
import joblib
from sklearn.preprocessing import StandardScaler

with mlflow.start_run(run_name="exercise_artifact_logging") as run:
    
    # ===== STEP 1: Train model with preprocessing =====
    print("\n🤖 STEP 1: Training model with preprocessing...\n")
    
    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train model (convert back to DataFrame for function)
    X_train_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
    X_test_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)
    
    model, metrics = train_random_forest(
        X_train_df, y_train, X_test_df, y_test,
        n_estimators=100,
        max_depth=10,
        random_state=42,
        log_to_mlflow=False
    )
    
    # Log metrics
    log_metrics_dict(metrics)
    
    # ===== ARTIFACT 1: Visualizations =====
    print("\n📊 ARTIFACT 1: Logging visualizations...\n")
    
    # Get predictions
    y_pred = model.predict(X_test_df)
    y_pred_proba = model.predict_proba(X_test_df)[:, 1]
    
    # Confusion Matrix
    fig_cm = plot_confusion_matrix(y_test, y_pred)
    mlflow.log_figure(fig_cm, "visualizations/confusion_matrix.png")
    plt.close(fig_cm)
    print("   ✅ Confusion matrix logged")
    
    # ROC Curve
    fig_roc = plot_roc_curve(y_test, y_pred_proba)
    mlflow.log_figure(fig_roc, "visualizations/roc_curve.png")
    plt.close(fig_roc)
    print("   ✅ ROC curve logged")
    
    # ===== ARTIFACT 2: Dataset Versioning =====
    print("\n🗂️  ARTIFACT 2: Dataset versioning...\n")
    
    # Calculate hash
    dataset_path = 'data/customer_churn.csv'
    with open(dataset_path, 'rb') as f:
        dataset_hash = hashlib.md5(f.read()).hexdigest()
    
    # Create metadata
    dataset_metadata = {
        'version': 'v1.0',
        'hash': dataset_hash,
        'collection_date': '2024-01-15',
        'total_rows': len(X_train) + len(X_test),
        'train_rows': len(X_train),
        'test_rows': len(X_test),
        'n_features': X_train.shape[1],
        'feature_names': X_train.columns.tolist(),
        'target_distribution': {
            'train_churn_rate': float(y_train.mean()),
            'test_churn_rate': float(y_test.mean()),
            'train_positive': int(y_train.sum()),
            'train_negative': int((y_train == 0).sum())
        },
        'data_source': 'CRM Database'
    }
    
    # Log as JSON artifact
    with open('dataset_metadata.json', 'w') as f:
        json.dump(dataset_metadata, f, indent=2)
    
    mlflow.log_artifact('dataset_metadata.json', artifact_path='datasets')
    os.remove('dataset_metadata.json')
    print("   ✅ Dataset metadata JSON logged")
    
    # Log key metadata as parameters
    mlflow.log_params({
        'dataset_version': dataset_metadata['version'],
        'dataset_hash': dataset_hash[:32],
        'total_rows': dataset_metadata['total_rows'],
        'n_features': dataset_metadata['n_features']
    })
    print("   ✅ Key metadata logged as parameters")
    
    # ===== ARTIFACT 3: Preprocessing Pipeline =====
    print("\n⚙️  ARTIFACT 3: Preprocessing pipeline...\n")
    
    # Save scaler
    scaler_path = 'preprocessing_scaler.pkl'
    joblib.dump(scaler, scaler_path)
    mlflow.log_artifact(scaler_path, artifact_path='preprocessing')
    os.remove(scaler_path)
    print("   ✅ Scaler saved and logged")
    
    # Save scaler metadata
    scaler_metadata = {
        'scaler_type': 'StandardScaler',
        'n_features': len(scaler.mean_),
        'feature_names': X_train.columns.tolist(),
        'mean_values': scaler.mean_.tolist(),
        'scale_values': scaler.scale_.tolist()
    }
    
    with open('scaler_metadata.json', 'w') as f:
        json.dump(scaler_metadata, f, indent=2)
    
    mlflow.log_artifact('scaler_metadata.json', artifact_path='preprocessing')
    os.remove('scaler_metadata.json')
    print("   ✅ Scaler metadata logged")
    
    # ===== Final Steps =====
    # Log model
    mlflow.sklearn.log_model(model, "model")
    print("\n💾 Model logged")
    
    # Add tag
    mlflow.set_tag('exercise', 'artifact_logging')
    print("🏷️  Tag added\n")
    
    run_id = run.info.run_id

print("="*70)
print("\n✅ EXERCISE COMPLETE!\n")
print(f"Run ID: {run_id}")

print("\n📦 Artifacts Logged:")
print("   visualizations/")
print("   ├── confusion_matrix.png")
print("   └── roc_curve.png")
print("   datasets/")
print("   └── dataset_metadata.json")
print("   preprocessing/")
print("   ├── preprocessing_scaler.pkl")
print("   └── scaler_metadata.json")

print("\n🎯 VERIFY in MLflow UI:")
print("   1. Find run 'exercise_artifact_logging'")
print("   2. Check Parameters: dataset_version, dataset_hash, etc.")
print("   3. Check Artifacts tab - see organized folder structure")
print("   4. Click through folders to view each artifact")
print("   5. Download any artifact to inspect locally")
print("   6. Check Tags: 'exercise' = 'artifact_logging'")

print("\n💡 Key Learnings:")
print("   ✓ Organized artifacts with folders")
print("   ✓ Comprehensive dataset versioning")
print("   ✓ Preprocessing pipeline preserved")
print("   ✓ Visualizations for analysis")
print("   ✓ Everything needed for reproducibility!")

---

## 🔍 MLflow UI: Artifact Navigation

Now that you've logged comprehensive artifacts, let's master navigating them in the MLflow UI.

### 🎯 Try This in MLflow UI: Complete Artifact Exploration

**Exercise: Navigate and Download Artifacts**

**Steps:**

1. **Find Your Run**
   - Go to MLflow UI (http://127.0.0.1:5000)
   - Click "Session5_Advanced_Tracking"
   - Find run "exercise_artifact_logging"
   - Click on the run name

2. **Explore Artifacts Tab**
   - Click "Artifacts" in the top navigation
   - You should see folders: model/, visualizations/, datasets/, preprocessing/

3. **View Visualizations**
   - Click visualizations/ folder
   - Click confusion_matrix.png
   - Image displays in the UI
   - Notice the download button (top right)
   - Click roc_curve.png - view it full screen

4. **Inspect Dataset Metadata**
   - Click datasets/ folder
   - Click dataset_metadata.json
   - JSON displays formatted in UI
   - Find: version, hash, feature_names, target_distribution

5. **Check Preprocessing**
   - Click preprocessing/ folder
   - See preprocessing_scaler.pkl (binary file)
   - Click scaler_metadata.json
   - See mean and scale values for each feature

6. **Download Artifacts**
   - Click any artifact
   - Click "Download" button (top right)
   - File downloads to your computer
   - Try opening the JSON files locally

7. **Compare Across Runs**
   - Go back to experiment view
   - Select 2-3 runs (checkboxes)
   - Click "Compare"
   - In Compare view, click "Artifacts"
   - See artifacts side-by-side

---

**🎓 What You Learned:**
- ✅ Navigate artifact directory structure
- ✅ View images directly in UI
- ✅ Inspect JSON/text files
- ✅ Download artifacts locally
- ✅ Compare artifacts across runs

**💡 Pro Tips:**
- Right-click images to "Open in new tab" for full size
- Use browser search (Ctrl+F) in JSON artifacts
- Download entire run: Click "Download Run" button
- Bookmark frequently used runs for quick access

---

⏱️ **Time for this exercise**: 5 minutes

💡 **Instructor Note**: Walk students through each step, emphasizing the ease of accessing any artifact months after the experiment.

---

### ✅ Part 3 Complete: Advanced Artifacts & Dataset Versioning

**🎉 Congratulations! You've Mastered:**

✅ **Dataset Versioning**
- Calculate and track dataset hashes
- Log comprehensive dataset metadata
- Track data lineage and transformations
- Prepare for reproducibility 6+ months later

✅ **Advanced Artifacts**
- Log preprocessing pipelines
- Organize artifacts in directories
- Create custom JSON configurations
- Document models with model cards

✅ **Production Patterns**
- Professional artifact organization
- Quality reports and configs
- Complete experiment documentation
- Everything needed for deployment

✅ **MLflow UI Mastery**
- Navigate complex artifact structures
- View and download any artifact
- Compare artifacts across runs
- Efficient artifact management

---

### 🚀 What's Next: Parts 4, 5, 6

**In the remaining sections, you'll learn:**

**Part 4**: Hyperparameter Tuning & Nested Runs
- Grid search with parent-child runs
- Compare 10+ hyperparameter combinations
- Find optimal configurations

**Part 5**: MLflow UI Mastery
- Advanced filtering (`metrics.accuracy > 0.9`)
- Run comparison techniques
- Parallel coordinates plots
- Tag-based organization

**Part 6**: Production Patterns
- Team collaboration workflows
- Git integration for full reproducibility
- Preparing for Model Registry (Session 6)
- Industry best practices

---

**💡 Remember**: Everything you've logged today will be used in Session 6 when we promote models to production through the Model Registry!

---

**Take a 5-minute break before continuing to Part 4!** ☕

---

# Part 4: Hyperparameter Tuning & Nested Runs

⏱️ **Time**: 30 minutes

## 🎯 The Hyperparameter Optimization Challenge

### The Real-World Problem

**Scenario**: You need to find the best Random Forest configuration.

**Hyperparameters to tune**:
- `n_estimators`: [50, 100, 200, 300]
- `max_depth`: [5, 10, 15, 20]
- `min_samples_split`: [2, 5, 10]
- `min_samples_leaf`: [1, 2, 4]

**Total combinations**: 4 × 4 × 3 × 3 = **144 experiments**

---

### ❌ The WRONG Way

Create 144 separate MLflow runs:

```python
# DON'T DO THIS!
for n_est in [50, 100, 200, 300]:
    for depth in [5, 10, 15, 20]:
        for split in [2, 5, 10]:
            for leaf in [1, 2, 4]:
                with mlflow.start_run():  # 144 independent runs
                    model = RandomForestClassifier(...)
                    # ...
```

**Problems**:
- ❌ No relationship between runs
- ❌ Can't find "grid search 1" easily
- ❌ No summary of best params
- ❌ Hard to compare entire searches
- ❌ Cluttered experiment view

---

### ✅ The RIGHT Way: Nested Runs

**Parent run**: The overall grid search  
**Child runs**: Each hyperparameter combination

```
Parent: Grid Search #1
├── Child: n_estimators=50, max_depth=5, ...
├── Child: n_estimators=50, max_depth=10, ...
├── Child: n_estimators=50, max_depth=15, ...
└── ... (144 children)
```

**Benefits**:
- ✅ Organized hierarchy
- ✅ Parent logs best params
- ✅ Compare child runs easily
- ✅ Clean experiment view
- ✅ Search-level metadata

---

### 🔑 Key Concepts

**Parent Run**:
- Overall search strategy
- Best parameters found
- Best score achieved
- Summary statistics

**Child Runs**:
- Individual hyperparameter combinations
- Training metrics
- Model artifacts (optional)
- Linked to parent

Let's see this in action! 🚀

In [ ]:
# Cell 43: Understanding nested runs structure

print("🏗️  Understanding Parent-Child Run Hierarchies\n")
print("="*70)

print("\n📊 STRUCTURE:\n")
print("Parent Run (Grid Search)")
print("│")
print("├── What it logs:")
print("│   • Search strategy (grid/random)")
print("│   • Parameter space explored")
print("│   • Best parameters found")
print("│   • Best score achieved")
print("│   • Number of combinations tried")
print("│   • Total search time")
print("│")
print("└── Child Runs (144 combinations)")
print("    ├── Child 1: {n_estimators: 50, max_depth: 5, ...}")
print("    │   → accuracy: 0.82")
print("    ├── Child 2: {n_estimators: 50, max_depth: 10, ...}")
print("    │   → accuracy: 0.85")
print("    ├── Child 3: {n_estimators: 100, max_depth: 5, ...}")
print("    │   → accuracy: 0.87")
print("    └── ... (141 more)")

print("\n" + "="*70)
print("\n🔑 HOW TO CREATE NESTED RUNS:\n")

print("Method 1: Explicit parent_run_id\n")
print("```python")
print("# Start parent")
print("with mlflow.start_run(run_name='grid_search_parent') as parent_run:")
print("    parent_run_id = parent_run.info.run_id")
print("    ")
print("    # Start children")
print("    for params in param_combinations:")
print("        with mlflow.start_run(")
print("            run_name=f'child_{i}',")
print("            nested=True  # This makes it a child!")
print("        ):")
print("            # Train and log...")
print("```")

print("\nMethod 2: Nested context managers\n")
print("```python")
print("# Parent context")
print("with mlflow.start_run(run_name='grid_search') as parent:")
print("    ")
print("    for params in param_combinations:")
print("        # Child context (automatically nested)")
print("        with mlflow.start_run(nested=True) as child:")
print("            # Train and log...")
print("```")

print("\n" + "="*70)
print("\n💡 BEST PRACTICES:\n")
print("1. Parent logs: search strategy, best params, best score")
print("2. Children log: individual params, metrics, models")
print("3. Use tags to mark parent vs child: 'role'='parent' or 'child'")
print("4. Limit child models: Only log best N models as artifacts")
print("5. Parent saves: best model as artifact")
print("6. Use descriptive names: 'grid_search_rf_2025' not 'run_1'")

print("\n✅ Let's implement this!")

In [ ]:
# Cell 44: Grid search with nested runs - complete example

print("🔬 Grid Search with Nested Runs\n")
print("="*70)

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
import time

# Define parameter grid (smaller for demo - 12 combinations)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, 20],
    'random_state': [42]
}

# Generate all combinations
param_combinations = list(ParameterGrid(param_grid))
print(f"🎯 Grid Search Configuration:")
print(f"   Parameter grid: {dict((k, v) for k, v in param_grid.items() if k != 'random_state')}")
print(f"   Total combinations: {len(param_combinations)}\n")

# Start parent run
print("🚀 Starting parent run...\n")

with mlflow.start_run(run_name="grid_search_random_forest") as parent_run:
    
    parent_run_id = parent_run.info.run_id
    
    # Log parent metadata
    mlflow.log_params({
        'search_type': 'grid',
        'algorithm': 'RandomForest',
        'total_combinations': len(param_combinations),
        'param_grid': str(param_grid)
    })
    
    mlflow.set_tags({
        'role': 'parent',
        'search_strategy': 'grid',
        'purpose': 'hyperparameter_tuning'
    })
    
    # Track best results
    best_score = 0
    best_params = None
    best_model = None
    
    start_time = time.time()
    
    # Iterate through combinations
    for i, params in enumerate(param_combinations, 1):
        
        # Start child run (nested=True makes it a child)
        with mlflow.start_run(
            run_name=f"rf_n{params['n_estimators']}_d{params['max_depth']}",
            nested=True
        ) as child_run:
            
            # Log child parameters
            mlflow.log_params(params)
            mlflow.set_tag('role', 'child')
            mlflow.set_tag('combination_number', i)
            
            # Train model
            model = RandomForestClassifier(**params)
            model.fit(X_train, y_train)
            
            # Evaluate
            train_score = model.score(X_train, y_train)
            test_score = model.score(X_test, y_test)
            
            # Log metrics
            mlflow.log_metrics({
                'train_accuracy': train_score,
                'test_accuracy': test_score,
                'train_test_gap': train_score - test_score
            })
            
            # Track best
            if test_score > best_score:
                best_score = test_score
                best_params = params
                best_model = model
            
            # Progress indicator
            if i % 4 == 0 or i == len(param_combinations):
                print(f"   Completed {i}/{len(param_combinations)} combinations... (best: {best_score:.4f})")
    
    search_time = time.time() - start_time
    
    # Log best results to parent
    print(f"\n✅ Grid search complete!\n")
    print(f"🏆 Best Results:")
    print(f"   Parameters: {best_params}")
    print(f"   Test Accuracy: {best_score:.4f}")
    print(f"   Search Time: {search_time:.2f}s\n")
    
    # Log to parent run
    mlflow.log_params({
        f'best_{k}': v for k, v in best_params.items()
    })
    
    mlflow.log_metrics({
        'best_test_accuracy': best_score,
        'search_time_seconds': search_time,
        'avg_time_per_combination': search_time / len(param_combinations)
    })
    
    # Log best model to parent
    mlflow.sklearn.log_model(best_model, "best_model")
    
    print(f"Parent Run ID: {parent_run_id}")

print("\n🎯 NOW: Check MLflow UI")
print("   1. Find 'grid_search_random_forest' run")
print("   2. Look for the expand arrow (▶) next to it")
print("   3. Click to expand and see all 12 child runs")
print("   4. Notice the hierarchy!")

### 🎯 Try This in MLflow UI: Explore Nested Runs

**Exercise: Navigate the Parent-Child Hierarchy**

**Steps:**

1. **Find the Parent Run**
   - Go to MLflow UI
   - Look for run "grid_search_random_forest"
   - Notice the **small arrow (▶)** to the left of the run name
   - This indicates it has child runs!

2. **Expand the Hierarchy**
   - Click the arrow (▶) to expand
   - See all 12 child runs appear indented below
   - Notice child runs are grouped under parent

3. **Inspect the Parent**
   - Click on the parent run name
   - **Parameters tab**: See `search_type`, `total_combinations`, `best_*` params
   - **Metrics tab**: See `best_test_accuracy`, `search_time_seconds`
   - **Tags tab**: See `role='parent'`
   - **Artifacts tab**: See `best_model/` (the winning model)

4. **Inspect a Child**
   - Click on any child run
   - **Parameters tab**: See specific `n_estimators`, `max_depth`
   - **Metrics tab**: See `train_accuracy`, `test_accuracy`
   - **Tags tab**: See `role='child'`, `combination_number`
   - Notice: No model artifact (saved space!)

5. **Compare Children**
   - Select multiple child runs (checkboxes)
   - Click "Compare" button
   - View parameters and metrics side-by-side
   - Sort by test_accuracy to find best

6. **Collapse/Expand**
   - Click the arrow again (▼) to collapse
   - Parent run stays visible, children hide
   - Clean experiment view!

---

**🎓 Key Observations:**

✅ **Organization**: Grid search is one logical unit  
✅ **Summary**: Parent shows overall results  
✅ **Details**: Children show individual attempts  
✅ **Clean UI**: Can collapse when not needed  
✅ **Efficient**: Only best model saved as artifact  

---

**💡 Pro Tip**: You can filter nested runs too!

Try: `tags.role = "child" AND metrics.test_accuracy > 0.85`

This shows only high-performing child runs!

---

⏱️ **Time for this exercise**: 5 minutes


In [ ]:
# Cell 46: Random search with nested runs

print("🎲 Random Search with Nested Runs\n")
print("="*70)

import numpy as np
from sklearn.ensemble import RandomForestClassifier

# Define parameter ranges
param_ranges = {
    'n_estimators': [50, 100, 150, 200, 250, 300],
    'max_depth': [5, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf': [1, 2, 4, 8]
}

n_iterations = 15  # Try 15 random combinations

print(f"🎯 Random Search Configuration:")
print(f"   Parameter ranges: {param_ranges}")
print(f"   Random samples: {n_iterations}\n")

# Set random seed for reproducibility
np.random.seed(42)

print("🚀 Starting random search...\n")

with mlflow.start_run(run_name="random_search_random_forest") as parent_run:
    
    parent_run_id = parent_run.info.run_id
    
    # Log parent metadata
    mlflow.log_params({
        'search_type': 'random',
        'algorithm': 'RandomForest',
        'n_iterations': n_iterations,
        'param_ranges': str(param_ranges)
    })
    
    mlflow.set_tags({
        'role': 'parent',
        'search_strategy': 'random',
        'purpose': 'hyperparameter_tuning'
    })
    
    # Track best results
    best_score = 0
    best_params = None
    best_model = None
    all_scores = []
    
    start_time = time.time()
    
    # Random iterations
    for i in range(n_iterations):
        
        # Sample random parameters
        params = {
            'n_estimators': np.random.choice(param_ranges['n_estimators']),
            'max_depth': np.random.choice(param_ranges['max_depth']),
            'min_samples_split': np.random.choice(param_ranges['min_samples_split']),
            'min_samples_leaf': np.random.choice(param_ranges['min_samples_leaf']),
            'random_state': 42
        }
        
        # Start child run
        with mlflow.start_run(
            run_name=f"random_{i+1}",
            nested=True
        ) as child_run:
            
            # Log child parameters
            mlflow.log_params(params)
            mlflow.set_tag('role', 'child')
            mlflow.set_tag('iteration_number', i+1)
            
            # Train model
            model = RandomForestClassifier(**params)
            model.fit(X_train, y_train)
            
            # Evaluate
            test_score = model.score(X_test, y_test)
            
            # Log metrics
            mlflow.log_metric('test_accuracy', test_score)
            
            all_scores.append(test_score)
            
            # Track best
            if test_score > best_score:
                best_score = test_score
                best_params = params
                best_model = model
                print(f"   🎯 New best! Iteration {i+1}: {test_score:.4f}")
            else:
                if (i+1) % 5 == 0:
                    print(f"   ⏳ Completed {i+1}/{n_iterations} iterations...")
    
    search_time = time.time() - start_time
    
    # Calculate statistics
    mean_score = np.mean(all_scores)
    std_score = np.std(all_scores)
    
    # Log summary to parent
    print(f"\n✅ Random search complete!\n")
    print(f"🏆 Best Results:")
    print(f"   Parameters: {best_params}")
    print(f"   Test Accuracy: {best_score:.4f}")
    print(f"   Mean Accuracy: {mean_score:.4f} (±{std_score:.4f})")
    print(f"   Search Time: {search_time:.2f}s\n")
    
    # Log to parent run
    mlflow.log_params({
        f'best_{k}': v for k, v in best_params.items() if k != 'random_state'
    })
    
    mlflow.log_metrics({
        'best_test_accuracy': best_score,
        'mean_test_accuracy': mean_score,
        'std_test_accuracy': std_score,
        'search_time_seconds': search_time
    })
    
    # Log best model
    mlflow.sklearn.log_model(best_model, "best_model")
    
    print(f"Parent Run ID: {parent_run_id}")

print("\n💡 Random vs Grid Search:")
print("   • Random: Explores parameter space more efficiently")
print("   • Grid: Exhaustive but can be computationally expensive")
print("   • Random often finds good solutions faster!")

In [ ]:
# Cell 47: Comparing Grid Search vs Random Search

print("⚖️  Grid Search vs Random Search: Which to Use?\n")
print("="*70)

comparison_data = {
    'Aspect': [
        'Coverage|',
        'Efficiency|',
        'Time Complexity|',
        'Best for|',
        'Disadvantage|',
        'Parallelization|',
        'Reproducibility|',
        'When to Use|'
    ],
    'Grid Search': [
        'Exhaustive - tests ALL combinations|',
        'Can be wasteful in high dimensions|',
        'Exponential: O(p₁ × p₂ × ... × pₙ)|',
        'Small parameter spaces (2-4 params)|',
        'Curse of dimensionality|',
        'Highly parallelizable|',
        'Fully deterministic|',
        'When you need to test ALL combinations|'
    ],
    'Random Search': [
        'Samples randomly - may miss optimal',
        'More efficient in high dimensions',
        'Linear: O(n_iterations)',
        'Large parameter spaces (5+ params)',
        'No guarantee of finding optimal',
        'Highly parallelizable',
        'Seed-dependent',
        'When parameter space is large'
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))
print("="*70)

print("\n📊 PRACTICAL EXAMPLE:\n")

print("Scenario: Tune 5 hyperparameters")
print("• Param 1: 10 values")
print("• Param 2: 10 values")
print("• Param 3: 5 values")
print("• Param 4: 5 values")
print("• Param 5: 4 values")
print("")
print("Grid Search: 10 × 10 × 5 × 5 × 4 = 10,000 combinations")
print("   → At 1 min per model = 166 hours (7 days!)\n")
print("Random Search: 100 iterations (1% of grid)")
print("   → At 1 min per model = 1.7 hours")
print("   → Often finds near-optimal solution! ✅\n")

print("="*70)
print("\n🎯 DECISION FRAMEWORK:\n")

print("USE GRID SEARCH when:")
print("   ✓ Small parameter space (< 1000 combinations)")
print("   ✓ You need to test every combination")
print("   ✓ Parameters have few discrete values")
print("   ✓ Example: n_estimators=[50,100,200], max_depth=[5,10,15]")
print("")

print("USE RANDOM SEARCH when:")
print("   ✓ Large parameter space (> 1000 combinations)")
print("   ✓ Many hyperparameters (5+)")
print("   ✓ Continuous parameter ranges")
print("   ✓ Limited compute time")
print("   ✓ Example: learning_rate, dropout, batch_size, layers, units, etc.")
print("")

print("HYBRID APPROACH (Best Practice):")
print("   1. Random search first (broad exploration)")
print("   2. Identify promising regions")
print("   3. Grid search second (fine-tune in that region)")
print("   Example:")
print("      • Random: learning_rate=[0.001-0.1], finds 0.01 works best")
print("      • Grid: learning_rate=[0.005, 0.01, 0.015, 0.02] (focused)")

print("\n💡 Advanced: In Module 2, we'll use Optuna for even smarter search!")
print("   → Bayesian optimization")
print("   → Learns from previous trials")
print("   → Often better than both grid and random")

In [ ]:
# Cell 48: Best practices for hyperparameter tuning with MLflow

print("⭐ Best Practices: Hyperparameter Tuning with MLflow\n")
print("="*70)

print("\n1️⃣  ORGANIZING RUNS\n")
print("✅ DO:")
print("   • Use descriptive parent names: 'grid_search_rf_v2_2025'")
print("   • Tag parent: role='parent', search_strategy='grid'")
print("   • Tag children: role='child', iteration=N")
print("   • Log search strategy in parent params")
print("")
print("❌ DON'T:")
print("   • Generic names: 'run_1', 'experiment_a'")
print("   • Mix multiple searches in one parent")
print("   • Forget to set nested=True for children")

print("\n" + "="*70)
print("\n2️⃣  WHAT TO LOG WHERE\n")
print("")
print("PARENT RUN should log:")
print("   ✓ Search strategy (grid/random/bayesian)")
print("   ✓ Parameter space explored")
print("   ✓ Best parameters found")
print("   ✓ Best score achieved")
print("   ✓ Total search time")
print("   ✓ Number of trials")
print("   ✓ BEST MODEL as artifact")
print("   ✓ Summary statistics (mean, std of scores)")
print("")
print("CHILD RUNS should log:")
print("   ✓ Individual hyperparameters")
print("   ✓ Training metrics")
print("   ✓ Validation metrics")
print("   ✓ Model (only for top N, or not at all to save space)")
print("   ✓ Training time for this combination")

print("\n" + "="*70)
print("\n3️⃣  SAVING SPACE\n")
print("")
print("Problem: 1000 models × 100MB each = 100GB of artifacts!")
print("")
print("Solutions:")
print("   ✓ Save only best model to parent")
print("   ✓ Save top-5 models to their child runs")
print("   ✓ For others: log params & metrics only")
print("   ✓ Use model registry (Session 6) for final model")
print("")
print("Example code:")
print("```python")
print("# Track top-5 models")
print("top_runs = sorted(all_runs, key=lambda x: x['score'], reverse=True)[:5]")
print("for run in top_runs:")
print("    # Go back and log model to this specific run")
print("    with mlflow.start_run(run_id=run['id']):")
print("        mlflow.sklearn.log_model(run['model'], 'model')")
print("```")

print("\n" + "="*70)
print("\n4️⃣  PARALLEL EXECUTION\n")
print("")
print("Both grid and random search are embarrassingly parallel!")
print("")
print("Options:")
print("   • joblib with n_jobs parameter")
print("   • Ray for distributed computing")
print("   • Kubernetes jobs (Module 2!)")
print("   • Cloud compute (AWS Batch, etc.)")
print("")
print("⚠️  Important: Each parallel worker needs nested=True!")

print("\n" + "="*70)
print("\n5️⃣  REPRODUCIBILITY\n")
print("")
print("Always log:")
print("   ✓ Random seeds (for random search)")
print("   ✓ Python version")
print("   ✓ Library versions (sklearn, numpy, etc.)")
print("   ✓ Dataset version (from Part 3!)")
print("   ✓ Git commit (from Session 4!)")
print("")
print("Example:")
print("```python")
print("mlflow.log_params({")
print("    'random_seed': 42,")
print("    'sklearn_version': sklearn.__version__,")
print("    'dataset_version': 'v1.0',")
print("    'git_commit': 'abc123'")
print("})")
print("```")

print("\n" + "="*70)
print("\n6️⃣  EARLY STOPPING\n")
print("")
print("For long searches, implement early stopping:")
print("   • Stop if no improvement after N iterations")
print("   • Stop if target metric achieved")
print("   • Stop if time budget exceeded")
print("")
print("Log early stopping info to parent run!")

print("\n✅ Follow these practices for professional hyperparameter tuning!")

### 🎯 Try This in MLflow UI: Compare 10+ Hyperparameter Runs

**Exercise: Find the Best Hyperparameters Using MLflow UI**

Now you have two parent runs with 27 child runs total (12 grid + 15 random). Let's find the best configuration!

---

#### Part A: Compare Grid Search Children

**Steps:**

1. **Expand grid search parent**
   - Find "grid_search_random_forest"
   - Click the arrow to expand
   - See all 12 child runs

2. **Select ALL grid search children**
   - Click the checkbox next to the parent (selects all children)
   - Or manually check all 12 child runs

3. **Compare the runs**
   - Click "Compare" button (top right)
   - You'll see a comparison table

4. **Find best parameters**
   - In the Metrics columns, sort by `test_accuracy` (descending)
   - Top row = best grid search configuration
   - Note the `n_estimators` and `max_depth` values

5. **Analyze patterns**
   - Do higher `n_estimators` always perform better?
   - What about `max_depth`?
   - Any overfitting? (Check `train_test_gap` metric)

---

#### Part B: Compare Random Search Children

**Steps:**

1. **Expand random search parent**
   - Find "random_search_random_forest"
   - Expand to see 15 child runs

2. **Select and compare**
   - Select all 15 child runs
   - Click "Compare"
   - Sort by `test_accuracy`

3. **Compare best from each search**
   - Which found better results: grid or random?
   - Look at the parameters of the top runs
   - Notice any patterns?

---

#### Part C: Compare Parent Runs

**Steps:**

1. **Collapse both parent runs**
   - Click arrows to hide children
   - Now you see just 2 runs

2. **Select both parents**
   - Check both "grid_search_random_forest" and "random_search_random_forest"
   - Click "Compare"

3. **Compare search strategies**
   - Look at `best_test_accuracy`: Which search found better model?
   - Look at `search_time_seconds`: Which was faster?
   - Compare `total_combinations` vs `n_iterations`

4. **Efficiency analysis**
   - Grid: Tested 12 combinations
   - Random: Tested 15 combinations
   - Which gave better accuracy per trial?

---

#### Part D: Create Parallel Coordinates Plot

**Steps:**

1. **Select multiple child runs**
   - From either parent, select 8-10 child runs
   - Mix high and low performers

2. **Open Compare view**
   - Click "Compare" button

3. **Switch to Parallel Coordinates**
   - Look for the "Parallel Coordinates Plot" button/tab
   - This creates an interactive visualization

4. **Interpret the plot**
   - Each line = one run
   - Vertical axes = parameters and metrics
   - Color intensity = metric value (usually)
   - Look for patterns:
     - Do best runs cluster on certain parameter values?
     - Which parameters seem most important?

---

**🎓 What You Should Discover:**

✅ **Grid search** is systematic but may miss good regions  
✅ **Random search** explores more broadly  
✅ Some parameters matter more than others  
✅ Parent runs give clean summary statistics  
✅ Child runs provide detailed exploration  
✅ **Parallel coordinates** reveal parameter interactions  

---

**💡 Pro Tips:**

- Use filters to narrow down: `metrics.test_accuracy > 0.85`
- Download comparison as CSV for analysis in Excel/Python
- Save good parameter combinations for future experiments
- Use the best run_id for Session 6 Model Registry!


In [ ]:
# Cell 50: Hands-on exercise - Implement your own grid search

print("🎯 HANDS-ON EXERCISE: Your Turn to Implement Grid Search!\n")
print("="*70)

print("\n📋 TASK:")
print("   Implement a grid search for Logistic Regression with nested runs")
print("")
print("🎯 REQUIREMENTS:")
print("")
print("   1. CREATE PARAMETER GRID:")
print("      • C: [0.01, 0.1, 1.0, 10.0]")
print("      • penalty: ['l1', 'l2']")
print("      • solver: ['liblinear'] (required for l1)")
print("      • max_iter: [1000]")
print("      Total: 8 combinations")
print("")
print("   2. PARENT RUN:")
print("      • Name: 'grid_search_logistic_regression'")
print("      • Log: search_type, algorithm, total_combinations")
print("      • Tags: role='parent', search_strategy='grid'")
print("")
print("   3. CHILD RUNS (for each combination):")
print("      • Name: f'logreg_C{C}_pen{penalty}'")
print("      • nested=True (CRITICAL!)")
print("      • Log: parameters (C, penalty, solver, max_iter)")
print("      • Log: test_accuracy metric")
print("      • Tags: role='child', combination_number=i")
print("")
print("   4. TRACK BEST MODEL:")
print("      • Keep track of best_score, best_params, best_model")
print("      • Log best results to parent")
print("      • Save best model as artifact in parent run")
print("")
print("   5. PARENT SUMMARY:")
print("      • Log best_C, best_penalty as params")
print("      • Log best_test_accuracy as metric")
print("      • Log search_time_seconds as metric")
print("")
print("   6. VERIFY IN MLFLOW UI:")
print("      • Parent run shows hierarchy (arrow)")
print("      • 8 child runs visible when expanded")
print("      • Best model saved to parent artifacts")
print("      • Can compare all 8 runs")

print("\n⏱️  Time: 10 minutes")

print("\n💡 Hints:")
print("   • Use sklearn.model_selection.ParameterGrid")
print("   • from sklearn.linear_model import LogisticRegression")
print("   • Remember nested=True in child runs!")
print("   • Track start time: start = time.time()")
print("   • Iterate: for i, params in enumerate(param_combinations, 1)")

print("\n" + "="*70)
print("\n📝 YOUR CODE BELOW (try it yourself first!)")
print("\n" + "="*70)
print("\n\n")

In [ ]:
# Cell 51: Exercise solution - Grid search for Logistic Regression

print("✅ SOLUTION: Grid Search for Logistic Regression\n")
print("="*70)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid
import time

# Define parameter grid
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'max_iter': [1000],
    'random_state': [42]
}

# Generate combinations
param_combinations = list(ParameterGrid(param_grid))

print(f"🎯 Starting Logistic Regression Grid Search")
print(f"   Total combinations: {len(param_combinations)}\n")

# Start parent run
with mlflow.start_run(run_name="grid_search_logistic_regression") as parent_run:
    
    # Log parent metadata
    mlflow.log_params({
        'search_type': 'grid',
        'algorithm': 'LogisticRegression',
        'total_combinations': len(param_combinations)
    })
    
    mlflow.set_tags({
        'role': 'parent',
        'search_strategy': 'grid',
        'purpose': 'hyperparameter_tuning'
    })
    
    # Track best results
    best_score = 0
    best_params = None
    best_model = None
    
    start_time = time.time()
    
    # Iterate through combinations
    for i, params in enumerate(param_combinations, 1):
        
        # Create descriptive name
        run_name = f"logreg_C{params['C']}_pen{params['penalty']}"
        
        # Start child run
        with mlflow.start_run(run_name=run_name, nested=True) as child_run:
            
            # Log parameters
            mlflow.log_params(params)
            mlflow.set_tag('role', 'child')
            mlflow.set_tag('combination_number', i)
            
            # Train model
            model = LogisticRegression(**params)
            model.fit(X_train, y_train)
            
            # Evaluate
            test_score = model.score(X_test, y_test)
            
            # Log metrics
            mlflow.log_metric('test_accuracy', test_score)
            
            # Track best
            if test_score > best_score:
                best_score = test_score
                best_params = params
                best_model = model
            
            print(f"   [{i}/{len(param_combinations)}] C={params['C']}, "
                  f"penalty={params['penalty']}: {test_score:.4f}")
    
    search_time = time.time() - start_time
    
    # Log best results to parent
    print(f"\n✅ Grid search complete!\n")
    print(f"🏆 Best Configuration:")
    print(f"   C: {best_params['C']}")
    print(f"   Penalty: {best_params['penalty']}")
    print(f"   Test Accuracy: {best_score:.4f}")
    print(f"   Search Time: {search_time:.2f}s\n")
    
    # Log to parent
    mlflow.log_params({
        'best_C': best_params['C'],
        'best_penalty': best_params['penalty']
    })
    
    mlflow.log_metrics({
        'best_test_accuracy': best_score,
        'search_time_seconds': search_time,
        'avg_time_per_combination': search_time / len(param_combinations)
    })
    
    # Save best model
    mlflow.sklearn.log_model(best_model, "best_model")
    
    parent_run_id = parent_run.info.run_id
    print(f"Parent Run ID: {parent_run_id}")

print("\n🎯 VERIFY IN MLFLOW UI:")
print("   1. Find 'grid_search_logistic_regression' parent run")
print("   2. Expand to see 8 child runs")
print("   3. Compare all children to find best manually")
print("   4. Check parent run for best_C and best_penalty params")
print("   5. Verify best_model artifact in parent")

print("\n💡 Key Learning:")
print("   L1 penalty (Lasso) vs L2 penalty (Ridge):")
print("   • L1: Feature selection (coefficients → 0)")
print("   • L2: Feature shrinkage (small coefficients)")
print("   Which performed better for this dataset?")

print("\n✅ EXERCISE COMPLETE!")

---

## 🚀 Advanced Topic: Bayesian Optimization (Preview)

### The Next Level of Hyperparameter Tuning

**Grid and Random Search limitations:**
- Don't learn from previous trials
- Treat all regions equally
- Can waste time on bad regions

**Bayesian Optimization** (Optuna, Hyperopt, etc.):
- ✅ Learns from past trials
- ✅ Focuses on promising regions
- ✅ Often finds better solutions faster
- ✅ Handles continuous and discrete params

---

### 📊 How It Works

```
Trial 1: Random guess → accuracy 0.75
Trial 2: Random guess → accuracy 0.82
Trial 3: Near Trial 2 (learned it's good!) → 0.85
Trial 4: Explore new region → 0.78
Trial 5: Back to good region → 0.87
...
```

**Exploitation**: Try parameters near past winners  
**Exploration**: Sometimes try unexplored regions  

---

### 🔮 Optuna with MLflow (Module 2 Preview)

```python
import optuna
from optuna.integration.mlflow import MLflowCallback

def objective(trial):
    # Optuna suggests params
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 5, 30)
    
    # Train and evaluate
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth
    )
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    
    return accuracy

# Optuna automatically logs to MLflow!
mlflc = MLflowCallback(
    tracking_uri="./mlruns",
    metric_name="accuracy"
)

study = optuna.create_study(direction='maximize')
study.optimize(
    objective,
    n_trials=50,
    callbacks=[mlflc]  # Auto-logs to MLflow!
)

print(f"Best params: {study.best_params}")
print(f"Best accuracy: {study.best_value}")
```

---

### 📈 Typical Performance Comparison

To find near-optimal hyperparameters:

| Method | Trials Needed | Time | Final Accuracy |
|--------|---------------|------|----------------|
| Grid Search | 1000+ | Days | 0.870 |
| Random Search | 100-200 | Hours | 0.875 |
| **Bayesian (Optuna)** | **50-75** | **Minutes** | **0.880** |

---

### 🎯 When to Use Each Method

**Grid Search**: 🔍
- Small parameter space
- Need comprehensive results
- Interpretability important

**Random Search**: 🎲
- Large parameter space
- Initial exploration
- Quick iteration needed

**Bayesian Optimization (Optuna)**: 🚀
- Complex parameter spaces
- Expensive training (each trial takes time)
- Need best results efficiently
- Production optimization

---

### 💡 Coming in Module 2: Modern Cloud-Native MLOps

We'll cover:
- **Optuna** for smart hyperparameter optimization
- **Kubernetes** for distributed grid searches
- **Airflow** for automated retraining with tuning
- **Kubeflow** for ML pipelines at scale

---

**For now**: Master grid and random search with MLflow nested runs!

You'll appreciate Optuna more after understanding these fundamentals. 🎓

In [ ]:
# Cell 54: Identify and save top runs for Session 6

print("💾 Preparing for Session 6: Model Registry\n")
print("="*70)

print("\n📋 In Session 6, we'll promote models to production using Model Registry.")
print("Let's identify the TOP 3 models from today's experiments!\n")

from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get current experiment
experiment = client.get_experiment_by_name("Session5_Advanced_Tracking")
experiment_id = experiment.experiment_id

print("🔍 Searching for best runs...\n")

# Search for all child runs (they have the actual models)
# We want child runs with test_accuracy metric
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.role = 'child' AND metrics.test_accuracy > 0",
    order_by=["metrics.test_accuracy DESC"],
    max_results=10  # Get top 10 to review
)

print(f"Found {len(runs)} child runs with test_accuracy metric\n")

# Display top 3
print("🏆 TOP 3 MODELS FROM SESSION 5:\n")
print("="*70)

top_runs = []

for i, run in enumerate(runs[:3], 1):
    run_id = run.info.run_id
    run_name = run.data.tags.get('mlflow.runName', 'Unnamed')
    accuracy = run.data.metrics.get('test_accuracy', 0)
    
    # Get parameters
    params = run.data.params
    
    print(f"\n#{i} - {run_name}")
    print(f"   Run ID: {run_id}")
    print(f"   Test Accuracy: {accuracy:.4f}")
    print(f"   Key Parameters:")
    
    # Show key params (limit to important ones)
    key_params = ['n_estimators', 'max_depth', 'C', 'penalty', 
                  'min_samples_split', 'min_samples_leaf']
    for param in key_params:
        if param in params:
            print(f"      {param}: {params[param]}")
    
    top_runs.append({
        'rank': i,
        'run_id': run_id,
        'run_name': run_name,
        'accuracy': accuracy,
        'params': dict(params)
    })

print("\n" + "="*70)

# Save to file for Session 6
import json

session5_best_runs = {
    'session': 5,
    'experiment_name': 'Session5_Advanced_Tracking',
    'experiment_id': experiment_id,
    'date_created': time.strftime('%Y-%m-%d'),
    'top_runs': top_runs
}

with open('session5_best_runs.json', 'w') as f:
    json.dump(session5_best_runs, f, indent=2)

print("\n✅ Top runs saved to: session5_best_runs.json")
print("   We'll use these run IDs in Session 6 for Model Registry!\n")

print("💡 What's Next in Session 6:")
print("   1. Register these top models in Model Registry")
print("   2. Add model descriptions and tags")
print("   3. Transition best model to 'Staging'")
print("   4. Run validation tests")
print("   5. Promote to 'Production'")
print("   6. Set up model serving")

print("\n📝 ACTION ITEM:")
print("   Keep session5_best_runs.json for Session 6!")
print("   It contains the run IDs of your best models.")

---

### ✅ Part 4 Complete: Hyperparameter Tuning & Nested Runs

**🎉 Congratulations! You've Mastered:**

✅ **Nested Run Hierarchies**
- Parent runs for overall search strategy
- Child runs for individual combinations
- Clean organization in MLflow UI
- Collapsible hierarchy for readability

✅ **Grid Search Implementation**
- Exhaustive parameter exploration
- Systematic testing of all combinations
- Best for small parameter spaces
- Easy to parallelize

✅ **Random Search Implementation**
- Efficient exploration of large spaces
- Better for high-dimensional problems
- Often finds good solutions faster
- Flexible iteration count

✅ **MLflow UI Navigation**
- Expand/collapse nested runs
- Compare 10+ runs simultaneously
- Sort and filter by metrics
- Identify best hyperparameters
- Parallel coordinates visualization

✅ **Best Practices**
- Proper tagging (role='parent'/'child')
- Save only best model to parent
- Log search metadata comprehensively
- Track search time and efficiency
- Prepare runs for Model Registry

---

### 🎯 Key Takeaways

1. **Nested runs** keep hyperparameter searches organized
2. **Parent runs** summarize, **child runs** provide details
3. **Grid search** for small spaces, **random search** for large
4. **Bayesian optimization** (Optuna) is next level - Module 2!
5. **Always save run IDs** of best models for Model Registry

---

### 📊 Your Accomplishments Today

You've run **multiple hyperparameter searches** with:
- 12 grid search combinations (Random Forest)
- 15 random search iterations (Random Forest)
- 8 grid search combinations (Logistic Regression)
- **Total: 35+ tracked experiments with proper hierarchy!**

All organized, comparable, and ready for production! 🚀

---

### 🚀 What's Next: Parts 5 & 6

**Part 5: MLflow UI Mastery** (Remaining)
- Advanced filtering queries
- Complex comparison techniques
- Parallel coordinates deep dive
- Tag-based organization
- Search operators and syntax

**Part 6: Production Patterns** (Remaining)
- Team collaboration workflows
- Git commit tracking (Session 4 integration)
- Reproducibility best practices
- Experiment naming conventions
- Preparing for Model Registry (Session 6)

---

**💡 Session 6 Preview**: You'll take these top 3 models and:
1. Register them in Model Registry
2. Add descriptions and versions
3. Transition through stages (Staging → Production)
4. Serve models via REST API
5. Implement A/B testing

Everything you've logged today becomes the foundation for production! 🎯

---

**Take a 5-minute break before continuing to Part 5!** ☕

**Or save your progress** - you've completed 75% of the session!

---

# Part 5: MLflow UI Mastery

⏱️ **Time**: 25 minutes

## 🎯 Becoming an MLflow UI Power User

You've logged dozens of runs. Now let's master finding exactly what you need, instantly.

### The Challenge

**Scenario**: Your team has 6 months of experiments:
- 2,500+ runs across 10 experiments
- Multiple models (RF, XGBoost, LogReg, Neural Networks)
- Different datasets (v1.0, v1.1, v2.0, v2.1)
- Various team members

**Questions you need to answer:**
- "Which Random Forest runs achieved accuracy > 0.90?"
- "Show me all runs from last week that used dataset v2.0"
- "Which runs trained by Arun beat the production baseline?"
- "Find experiments with low train-test gap (< 0.05)"
- "What's the best model for each algorithm type?"

### 🔍 The Solution: Advanced Filtering

MLflow UI has a **powerful query language** for filtering runs:

```
metrics.accuracy > 0.9
params.dataset_version = "v2.0"
tags.developer = "arun"
metrics.train_test_gap < 0.05
```

And you can **combine** them:

```
metrics.accuracy > 0.9 AND params.model_type = "RandomForest" AND tags.purpose = "production_candidate"
```

### 📋 What We'll Master

1. **Filter Syntax**: metrics, params, tags, attributes
2. **Operators**: `=`, `!=`, `<`, `>`, `<=`, `>=`, `LIKE`, `ILIKE`
3. **Logical Operators**: `AND`, `OR`, `NOT`
4. **Advanced Patterns**: Combining multiple conditions
5. **Sorting**: Order by any metric/param
6. **Comparison**: Side-by-side analysis
7. **Export**: Download for further analysis

Let's become MLflow UI power users! 🚀

In [ ]:
# Cell 56: MLflow filtering syntax reference

print("🔍 MLflow UI Filter Syntax Reference\n")
print("="*70)

print("\n📊 FILTER TYPES:\n")

print("1. METRICS (numeric performance measures)")
print("   Syntax: metrics.<metric_name> <operator> <value>")
print("   Examples:")
print("   • metrics.accuracy > 0.85")
print("   • metrics.f1_score >= 0.80")
print("   • metrics.loss < 0.5")
print("   • metrics.roc_auc != 0.0")
print("")

print("2. PARAMETERS (hyperparameters and configs)")
print("   Syntax: params.<param_name> = '<value>'")
print("   Examples:")
print("   • params.n_estimators = '100'")
print("   • params.model_type = 'RandomForest'")
print("   • params.dataset_version = 'v2.0'")
print("   ⚠️  Note: Parameters are strings, use quotes!")
print("")

print("3. TAGS (metadata and labels)")
print("   Syntax: tags.<tag_name> = '<value>'")
print("   Examples:")
print("   • tags.purpose = 'production'")
print("   • tags.developer = 'arun'")
print("   • tags.role = 'parent'")
print("   • tags.experiment_type = 'baseline'")
print("")

print("4. ATTRIBUTES (run metadata)")
print("   Syntax: attributes.<attribute_name>")
print("   Examples:")
print("   • attributes.status = 'FINISHED'")
print("   • attributes.run_name = 'grid_search_rf'")
print("   • attributes.start_time > '2025-10-01'")
print("")

print("="*70)
print("\n🔢 OPERATORS:\n")

operators = [
    ('=', 'Equal to', "params.model = 'RF'"),
    ('!=', 'Not equal to', "metrics.accuracy != 0.0"),
    ('>', 'Greater than', "metrics.accuracy > 0.85"),
    ('<', 'Less than', "metrics.loss < 0.5"),
    ('>=', 'Greater or equal', "metrics.f1 >= 0.80"),
    ('<=', 'Less or equal', "metrics.train_time <= 60"),
    ('LIKE', 'Pattern match (case-sensitive)', "params.model_type LIKE '%Forest%'"),
    ('ILIKE', 'Pattern match (case-insensitive)', "tags.developer ILIKE '%arun%'")
]

for op, desc, example in operators:
    print(f"   {op:6s} - {desc:30s} - {example}")

print("\n" + "="*70)
print("\n🔗 LOGICAL OPERATORS:\n")

print("   AND - Both conditions must be true")
print("         metrics.accuracy > 0.85 AND params.model = 'RandomForest'")
print("")
print("   OR  - At least one condition must be true")
print("         params.model = 'RandomForest' OR params.model = 'XGBoost'")
print("")
print("   NOT - Negate a condition")
print("         NOT tags.purpose = 'test'")
print("")

print("="*70)
print("\n💡 PRO TIPS:\n")
print("   • Parameter values are strings: params.n_estimators = '100' (not 100)")
print("   • Use parentheses for complex logic: (A AND B) OR (C AND D)")
print("   • LIKE uses % as wildcard: LIKE '%test%' matches anything with 'test'")
print("   • Combine multiple filters for precision")
print("   • Sort results: Click column headers in UI")

print("\n✅ Now let's use these filters in practice!")

### 🎯 Try This in MLflow UI: Advanced Filtering

**Exercise: Filter Your Runs Using Query Language**

Open MLflow UI and try these filters in the **Search Runs** box:

---

#### 🔍 Filter 1: High-Performing Models

**Query**:
```
metrics.test_accuracy > 0.85
```

**What you should see**: Only runs with test accuracy above 85%

**Questions to explore**:
- How many runs meet this criteria?
- What model types appear most?
- Click on column headers to sort

---

#### 🔍 Filter 2: Random Forest with Good Performance

**Query**:
```
params.n_estimators = '100' AND metrics.test_accuracy > 0.84
```

**What you should see**: Random Forest runs (n_estimators=100) with accuracy > 84%

**Try variations**:
- Change n_estimators to '200'
- Adjust accuracy threshold
- Add more conditions

---

#### 🔍 Filter 3: Child Runs from Hyperparameter Search

**Query**:
```
tags.role = 'child' AND metrics.test_accuracy > 0.83
```

**What you should see**: Only child runs (from nested searches) with good performance

**Insight**: These are individual hyperparameter combinations that worked well!

---

#### 🔍 Filter 4: Find Overfitting

**Query**:
```
metrics.train_test_gap > 0.05
```

**What you should see**: Runs where training accuracy significantly exceeds test accuracy

**Interpretation**: These models might be overfitting! Consider:
- Regularization
- Less complex models
- More training data

---

#### 🔍 Filter 5: Parent Runs Only

**Query**:
```
tags.role = 'parent'
```

**What you should see**: Only parent runs (grid searches, random searches)

**Why this is useful**: Clean view of overall search strategies without seeing all children

---

#### 🔍 Filter 6: Complex Multi-Condition

**Query**:
```
metrics.test_accuracy > 0.85 AND tags.role = 'child' AND params.max_depth = '10'
```

**What you should see**: High-performing child runs with max_depth=10

**This tells you**: This hyperparameter value consistently leads to good results!

---

### 💡 Pro Tips

1. **Save Filters**: Bookmark URLs with filters applied
2. **Clear Filters**: Click the X to reset
3. **Sort**: Click any column header
4. **Multi-sort**: Hold Shift, click multiple columns
5. **Export**: Select runs → Download CSV

---

### 🎯 Challenge: Create Your Own Filter

Find:
- Logistic Regression runs
- With C parameter between 0.1 and 10
- Test accuracy > 0.82
- That are child runs

**Hint**: You'll need 4 conditions with AND

```
params.C >= '0.1' AND params.C <= '10' AND metrics.test_accuracy > 0.82 AND tags.role = 'child'
```

---

⏱️ **Time**: 8-10 minutes


In [ ]:
# Cell 58: Programmatic search using MlflowClient

print("🔍 Programmatic Run Search with MlflowClient\n")
print("="*70)

from mlflow.tracking import MlflowClient
import pandas as pd

client = MlflowClient()

# Get experiment
experiment = client.get_experiment_by_name("Session5_Advanced_Tracking")
experiment_id = experiment.experiment_id

print("📊 Example 1: Find High-Performing Runs\n")

# Search for runs with good accuracy
high_performing_runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="metrics.test_accuracy > 0.85",
    order_by=["metrics.test_accuracy DESC"],
    max_results=5
)

print(f"Found {len(high_performing_runs)} runs with accuracy > 0.85\n")

for i, run in enumerate(high_performing_runs, 1):
    run_name = run.data.tags.get('mlflow.runName', 'Unnamed')
    accuracy = run.data.metrics.get('test_accuracy', 0)
    print(f"{i}. {run_name:40s} - Accuracy: {accuracy:.4f}")

print("\n" + "="*70)
print("\n📊 Example 2: Complex Query\n")

# Complex filter: Child runs, high accuracy, specific params
complex_query = """
    tags.role = 'child' AND 
    metrics.test_accuracy > 0.84 AND
    params.max_depth = '10'
"""

filtered_runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string=complex_query,
    order_by=["metrics.test_accuracy DESC"]
)

print(f"Query: {complex_query}")
print(f"\nFound {len(filtered_runs)} matching runs\n")

if len(filtered_runs) > 0:
    for run in filtered_runs[:3]:
        run_name = run.data.tags.get('mlflow.runName', 'Unnamed')
        accuracy = run.data.metrics.get('test_accuracy', 0)
        n_est = run.data.params.get('n_estimators', 'N/A')
        print(f"   {run_name:35s} - Acc: {accuracy:.4f}, n_est: {n_est}")
else:
    print("   No runs matched these criteria")

print("\n" + "="*70)
print("\n📊 Example 3: Export to DataFrame\n")

# Get all child runs and convert to DataFrame
all_child_runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="tags.role = 'child'",
    max_results=100
)

# Extract data for DataFrame
data = []
for run in all_child_runs:
    data.append({
        'run_id': run.info.run_id,
        'run_name': run.data.tags.get('mlflow.runName', 'Unnamed'),
        'test_accuracy': run.data.metrics.get('test_accuracy', None),
        'n_estimators': run.data.params.get('n_estimators', None),
        'max_depth': run.data.params.get('max_depth', None),
        'start_time': run.info.start_time
    })

df_runs = pd.DataFrame(data)

if len(df_runs) > 0:
    print(f"Exported {len(df_runs)} child runs to DataFrame\n")
    print("Top 5 by accuracy:")
    print(df_runs.nlargest(5, 'test_accuracy')[['run_name', 'test_accuracy', 'n_estimators', 'max_depth']])
    
    print("\n💾 You can save this to CSV:")
    print("   df_runs.to_csv('experiment_results.csv', index=False)")
else:
    print("No child runs found")

print("\n" + "="*70)
print("\n💡 USE CASES FOR PROGRAMMATIC SEARCH:\n")
print("   1. Automated model selection (find best, deploy)")
print("   2. Generate reports for stakeholders")
print("   3. Data analysis in Jupyter/Python scripts")
print("   4. CI/CD pipelines (validate performance)")
print("   5. Dashboards and monitoring tools")
print("   6. Alerting (if no runs > threshold, alert team)")

print("\n✅ MlflowClient gives you full programmatic control!")

In [ ]:
# Cell 59: Advanced run comparison and visualization

print("📊 Advanced Run Comparison\n")
print("="*70)

from mlflow.tracking import MlflowClient
import pandas as pd
import matplotlib.pyplot as plt

client = MlflowClient()
experiment = client.get_experiment_by_name("Session5_Advanced_Tracking")
experiment_id = experiment.experiment_id

print("🔍 Fetching data for visualization...\n")

# Get all runs with test_accuracy
runs = client.search_runs(
    experiment_ids=[experiment_id],
    filter_string="metrics.test_accuracy > 0 AND tags.role = 'child'",
    max_results=50
)

print(f"Found {len(runs)} runs for analysis\n")

if len(runs) > 0:
    # Extract data
    data = []
    for run in runs:
        data.append({
            'run_name': run.data.tags.get('mlflow.runName', 'Unnamed')[:30],
            'accuracy': run.data.metrics.get('test_accuracy', 0),
            'n_estimators': int(run.data.params.get('n_estimators', 0) or 0),
            'max_depth': int(run.data.params.get('max_depth', 0) or 0),
        })
    
    df = pd.DataFrame(data)
    df = df[df['n_estimators'] > 0]  # Filter valid runs
    
    if len(df) > 0:
        # Create visualizations
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Plot 1: Accuracy distribution
        axes[0].hist(df['accuracy'], bins=15, edgecolor='black', alpha=0.7)
        axes[0].axvline(df['accuracy'].mean(), color='red', linestyle='--', 
                       linewidth=2, label=f'Mean: {df["accuracy"].mean():.3f}')
        axes[0].set_xlabel('Test Accuracy')
        axes[0].set_ylabel('Number of Runs')
        axes[0].set_title('Distribution of Test Accuracy Across Runs')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
        
        # Plot 2: Parameter vs Accuracy
        scatter = axes[1].scatter(df['n_estimators'], df['accuracy'], 
                                  c=df['max_depth'], cmap='viridis', 
                                  s=100, alpha=0.6, edgecolors='black')
        axes[1].set_xlabel('n_estimators')
        axes[1].set_ylabel('Test Accuracy')
        axes[1].set_title('Hyperparameter Impact on Accuracy')
        axes[1].grid(True, alpha=0.3)
        
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=axes[1])
        cbar.set_label('max_depth')
        
        plt.tight_layout()
        plt.savefig('experiment_analysis.png', dpi=150, bbox_inches='tight')
        print("✅ Visualization saved: experiment_analysis.png\n")
        plt.show()
        
        # Summary statistics
        print("="*70)
        print("\n📈 EXPERIMENT SUMMARY STATISTICS:\n")
        print(f"Total Runs Analyzed: {len(df)}")
        print(f"Mean Accuracy: {df['accuracy'].mean():.4f}")
        print(f"Std Accuracy: {df['accuracy'].std():.4f}")
        print(f"Best Accuracy: {df['accuracy'].max():.4f}")
        print(f"Worst Accuracy: {df['accuracy'].min():.4f}")
        print(f"Median Accuracy: {df['accuracy'].median():.4f}")
        
        # Best configurations
        print("\n🏆 TOP 3 CONFIGURATIONS:\n")
        top3 = df.nlargest(3, 'accuracy')
        for i, row in enumerate(top3.itertuples(), 1):
            print(f"{i}. {row.run_name}")
            print(f"   Accuracy: {row.accuracy:.4f}, n_estimators: {row.n_estimators}, max_depth: {row.max_depth}")
        
        print("\n" + "="*70)
    else:
        print("Not enough valid runs for visualization")
else:
    print("No runs found for visualization")

print("\n💡 INSIGHTS FROM VISUALIZATION:")
print("   • Histogram shows if most models perform similarly or vary widely")
print("   • Scatter plot reveals parameter-performance relationships")
print("   • Color (max_depth) shows if deeper trees help")
print("   • Summary stats give objective performance measures")

print("\n🎯 ACTIONABLE DECISIONS:")
print("   1. If tight distribution → hyperparams don't matter much")
print("   2. If wide distribution → hyperparameter tuning is critical")
print("   3. Scatter plot patterns → which params to focus on")
print("   4. Top 3 configs → candidates for production")

---

### ✅ Part 5 Complete: MLflow UI Mastery

**🎉 You've Mastered:**

✅ **Advanced Filter Syntax**
- metrics, params, tags, attributes
- Comparison operators (=, !=, <, >, <=, >=)
- Pattern matching (LIKE, ILIKE)
- Logical operators (AND, OR, NOT)

✅ **Complex Query Patterns**
- Multi-condition filtering
- Finding high-performing runs
- Detecting overfitting
- Time-based queries

✅ **Programmatic Access**
- MlflowClient.search_runs()
- filter_string and order_by
- Export to DataFrame
- Save to CSV for analysis

✅ **Data Analysis & Visualization**
- Accuracy distribution histograms
- Hyperparameter impact scatter plots
- Summary statistics calculation
- Top configuration identification

---

### 🎯 Real-World Applications

You can now:
- **Find best models** instantly from thousands of runs
- **Detect issues** like overfitting or poor performance
- **Analyze patterns** in hyperparameter effectiveness
- **Generate reports** for stakeholders
- **Automate decisions** in CI/CD pipelines
- **Export data** for deeper analysis

---

### 💡 Key Takeaway

**MLflow UI is not just a viewer - it's a powerful query engine** for your ML experiments!

Combine:
- UI filtering for quick exploration
- Programmatic search for automation
- Data export for deep analysis

---

**Next up**: Part 6 - Production patterns and preparing for Session 6! 🚀

---

# Part 6: Production Patterns & Best Practices

⏱️ **Time**: 20 minutes

## 🎯 From Experiments to Production

You've learned all the technical skills. Now let's learn how to use them **in production teams**.

### The Reality of Production ML

**In production, you need:**
- Team members understanding each other's experiments
- Reproducible results 6 months later
- Clear audit trails for compliance
- Smooth handoff to Model Registry
- Collaboration without conflicts

### 📋 What We'll Cover

1. **Team Collaboration**: Naming, tagging, organization
2. **Reproducibility**: What to log for 100% reproduction
3. **Git Integration**: Link experiments to code versions
4. **Experiment Organization**: Strategies for scale
5. **Session 6 Preparation**: Model Registry handoff

Let's build production-grade habits! 🏗️

In [ ]:
# Cell 62: Team collaboration best practices

print("👥 Team Collaboration with MLflow\n")
print("="*70)

print("\n1️⃣  EXPERIMENT NAMING CONVENTIONS\n")
print("✅ GOOD NAMING PATTERNS:\n")

good_names = [
    ("[Developer]_[Model]_[Date]", "sarah_rf_churn_2025-10-29"),
    ("[Project]_[Phase]_[Version]", "churn_baseline_v1"),
    ("[Team]_[Task]_[Iteration]", "ml_team_hypertuning_run3"),
    ("[Dataset]_[Model]_[Purpose]", "churn_v2_xgb_production"),
]

for pattern, example in good_names:
    print(f"   Pattern: {pattern:30s}")
    print(f"   Example: {example}\n")

print("❌ BAD NAMING PATTERNS:\n")
bad_names = [
    ("test", "Too generic, no context"),
    ("experiment_1", "No semantic meaning"),
    ("my_run", "Not descriptive"),
    ("final_final_v2", "Unclear versioning"),
]

for name, reason in bad_names:
    print(f"   ❌ '{name}' - {reason}")

print("\n" + "="*70)
print("\n2️⃣  TAG STRATEGIES FOR TEAMS\n")

print("ESSENTIAL TAGS (always include):\n")
essential_tags = [
    ('developer', 'arun', 'Who ran this experiment'),
    ('purpose', 'baseline/tuning/production', 'Why this run exists'),
    ('project', 'customer_churn', 'Which business problem'),
    ('status', 'experimental/validated/production', 'Readiness level'),
]

for tag, example, desc in essential_tags:
    print(f"   • {tag:12s}: '{example:25s}' - {desc}")

print("\nOPTIONAL TAGS (add when relevant):\n")
optional_tags = [
    ('ticket_id', 'JIRA-1234', 'Link to tracking ticket'),
    ('environment', 'staging/production', 'Where this will deploy'),
    ('priority', 'high/medium/low', 'Business importance'),
    ('review_status', 'pending/approved', 'Review workflow'),
]

for tag, example, desc in optional_tags:
    print(f"   • {tag:15s}: '{example:20s}' - {desc}")

print("\n" + "="*70)
print("\n3️⃣  EXPERIMENT ORGANIZATION\n")

print("STRATEGY 1: By Project")
print("   • Experiment: 'customer_churn_prediction'")
print("   • Experiment: 'fraud_detection'")
print("   • Experiment: 'product_recommendations'")
print("   → One experiment per business use case\n")

print("STRATEGY 2: By Phase")
print("   • Experiment: 'churn_baseline'")
print("   • Experiment: 'churn_hypertuning'")
print("   • Experiment: 'churn_production'")
print("   → Track project evolution\n")

print("STRATEGY 3: By Team")
print("   • Experiment: 'ml_team_experiments'")
print("   • Experiment: 'research_team_experiments'")
print("   • Experiment: 'data_science_explorations'")
print("   → Organize by ownership\n")

print("💡 RECOMMENDATION: Combine strategies!")
print("   Example: 'ml_team_churn_baseline_2024Q4'\n")

print("="*70)
print("\n4️⃣  RUN DESCRIPTIONS\n")

print("Always add descriptions (via mlflow.set_tag('mlflow.note.content', ...)):\n")

example_desc = '''Testing Random Forest with:
- Dataset: churn_v2.0 (10K samples, 5% churn rate)
- Feature engineering: Added tenure_months and avg_spend
- Goal: Beat baseline accuracy of 0.83
- Context: JIRA-1234 - Improve churn prediction
- Next steps: If successful, deploy to staging
'''

print(f"   {example_desc}")

print("\n💡 Good descriptions help team members understand your work!\n")

print("="*70)
print("\n5️⃣  CODE EXAMPLE: Team-Ready Run\n")

example_code = '''# Team-ready MLflow run
with mlflow.start_run(run_name="sarah_rf_churn_2025-10-29") as run:
    
    # Essential tags
    mlflow.set_tags({
        'developer': 'arun',
        'purpose': 'hyperparameter_tuning',
        'project': 'customer_churn',
        'status': 'experimental',
        'ticket_id': 'JIRA-1234'
    })
    
    # Description
    mlflow.set_tag('mlflow.note.content', 
                   'Testing Random Forest with new features. Goal: >0.85 accuracy')
    
    # Your training code...
    # ...
'''

print(example_code)

print("\n✅ Now your team knows: who, what, why, and status!")

In [ ]:
# Cell 63: Complete reproducibility checklist

print("🔄 Complete Reproducibility Checklist\n")
print("="*70)

print("\n✅ WHAT TO LOG FOR 100% REPRODUCIBILITY\n")

print("1️⃣  CODE VERSION (Critical!)\n")
print("   • Git commit SHA")
print("   • Git branch name")
print("   • Git repository URL")
print("   • Code snippet or script path\n")
print("   Example:")
print("   mlflow.log_param('git_commit', 'abc123def456')")
print("   mlflow.log_param('git_branch', 'feature/churn-improvement')")
print("   mlflow.set_tag('git_repo', 'github.com/company/ml-models')\n")

print("2️⃣  DATA VERSION (Session 3 Part 3!)\n")
print("   • Dataset version ID")
print("   • Dataset hash (MD5/SHA256)")
print("   • Data collection date")
print("   • Number of rows/features")
print("   • Target distribution\n")
print("   Example:")
print("   mlflow.log_param('dataset_version', 'v2.0')")
print("   mlflow.log_param('dataset_hash', 'a1b2c3d4...')")
print("   mlflow.log_param('dataset_rows', '10000')\n")

print("3️⃣  ENVIRONMENT\n")
print("   • Python version")
print("   • Key library versions (sklearn, numpy, pandas, mlflow)")
print("   • OS/platform")
print("   • Hardware (CPU/GPU)\n")
print("   Example:")
print("   import sys, sklearn, numpy, pandas, mlflow")
print("   mlflow.log_param('python_version', sys.version)")
print("   mlflow.log_param('sklearn_version', sklearn.__version__)")
print("   mlflow.log_param('numpy_version', numpy.__version__)\n")

print("4️⃣  RANDOM SEEDS\n")
print("   • All random seeds used")
print("   • Python random seed")
print("   • Numpy random seed")
print("   • Model random_state\n")
print("   Example:")
print("   mlflow.log_param('random_seed', 42)")
print("   mlflow.log_param('numpy_seed', 42)")
print("   mlflow.log_param('model_random_state', 42)\n")

print("5️⃣  PREPROCESSING STEPS\n")
print("   • Feature engineering applied")
print("   • Scaling method")
print("   • Missing value handling")
print("   • Feature selection\n")
print("   Example:")
print("   mlflow.log_param('scaling', 'StandardScaler')")
print("   mlflow.log_param('missing_values', 'mean_imputation')\n")

print("6️⃣  MODEL CONFIGURATION\n")
print("   • All hyperparameters")
print("   • Model architecture (for neural networks)")
print("   • Training configuration (epochs, batch size)\n")

print("7️⃣  TRAINING DETAILS\n")
print("   • Train/test split ratio")
print("   • Cross-validation strategy")
print("   • Training duration")
print("   • Number of iterations/epochs\n")

print("="*70)
print("\n🎯 REPRODUCIBILITY HELPER FUNCTION\n")

helper_code = '''import sys
import sklearn
import numpy as np
import pandas as pd
import mlflow
import subprocess

def log_reproducibility_info():
    """Log all information needed for reproducibility"""
    
    # Environment
    mlflow.log_param('python_version', f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")
    mlflow.log_param('sklearn_version', sklearn.__version__)
    mlflow.log_param('numpy_version', np.__version__)
    mlflow.log_param('pandas_version', pd.__version__)
    mlflow.log_param('mlflow_version', mlflow.__version__)
    
    # Git info (if in a git repo)
    try:
        git_commit = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD']
        ).decode('utf-8').strip()
        mlflow.log_param('git_commit', git_commit[:8])
        
        git_branch = subprocess.check_output(
            ['git', 'rev-parse', '--abbrev-ref', 'HEAD']
        ).decode('utf-8').strip()
        mlflow.log_param('git_branch', git_branch)
    except:
        print("Not in a git repository")
    
    # Timestamp
    from datetime import datetime
    mlflow.log_param('logged_at', datetime.now().isoformat())
    
    print("✅ Reproducibility info logged!")

# Usage:
with mlflow.start_run():
    log_reproducibility_info()
    # ... rest of your training code
'''

print(helper_code)

print("\n💡 Use this function at the start of every training run!")

print("="*70)
print("\n🎓 GOLDEN RULE OF REPRODUCIBILITY:\n")
print("   'If you can't reproduce it, it didn't happen.'")
print("\n   Log EVERYTHING needed to recreate the exact same result.")
print("   Your future self (and your team) will thank you!")

In [ ]:
# Cell 64: Git integration with MLflow

print("🔗 Git Integration: Linking Code to Experiments\n")
print("="*70)

print("\n💡 WHY LINK MLFLOW TO GIT?\n")
print("   Scenario: Model performance drops in production\n")
print("   With Git integration:")
print("   1. Check MLflow run → get git commit SHA")
print("   2. Checkout that exact commit: git checkout abc123")
print("   3. See EXACT code that trained the model")
print("   4. Reproduce or debug easily!\n")
print("   Without Git integration:")
print("   ❌ 'Which version of the code was this?'")
print("   ❌ 'Did we have that feature engineering step?'")
print("   ❌ Hours wasted searching...\n")

print("="*70)
print("\n🔧 METHOD 1: Manual Git Logging\n")

manual_code = '''import subprocess
import mlflow

def get_git_info():
    """Extract current Git information"""
    try:
        # Get commit SHA
        commit_sha = subprocess.check_output(
            ['git', 'rev-parse', 'HEAD']
        ).decode('utf-8').strip()
        
        # Get branch name
        branch = subprocess.check_output(
            ['git', 'rev-parse', '--abbrev-ref', 'HEAD']
        ).decode('utf-8').strip()
        
        # Get repo URL
        repo_url = subprocess.check_output(
            ['git', 'config', '--get', 'remote.origin.url']
        ).decode('utf-8').strip()
        
        return {
            'git_commit': commit_sha[:8],  # Short SHA
            'git_commit_full': commit_sha,
            'git_branch': branch,
            'git_repo': repo_url
        }
    except subprocess.CalledProcessError:
        return None

# Use in MLflow run:
with mlflow.start_run():
    git_info = get_git_info()
    if git_info:
        mlflow.log_params(git_info)
        mlflow.set_tag('git_link', 
                       f"{git_info['git_repo']}/commit/{git_info['git_commit_full']}")
        print(f"✅ Logged Git commit: {git_info['git_commit']}")
    else:
        print("⚠️  Not in a Git repository")
'''

print(manual_code)

print("\n" + "="*70)
print("\n🔧 METHOD 2: Automatic Git Logging (MLflow Feature)\n")

print("MLflow can automatically log Git info if you:")
print("1. Use MLflow Projects (we'll cover in Session 6)")
print("2. Set environment variable: MLFLOW_GIT_COMMIT\n")

auto_code = '''# In your CI/CD pipeline or script:
import os
import subprocess

# Set Git commit as environment variable
git_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('utf-8').strip()
os.environ['MLFLOW_GIT_COMMIT'] = git_commit

# Now MLflow automatically includes it!
with mlflow.start_run():
    # Git info logged automatically
    pass
'''

print(auto_code)

print("\n" + "="*70)
print("\n💡 BEST PRACTICES:\n")

print("1. **Always log Git commit SHA**")
print("   - Critical for reproducibility")
print("   - Links experiments to exact code\n")

print("2. **Log branch name**")
print("   - Know if this was main, feature, or hotfix")
print("   - Helps understand context\n")

print("3. **Create clickable links**")
print("   - Store GitHub/GitLab URL with commit")
print("   - Example: 'https://github.com/org/repo/commit/abc123'")
print("   - Easy navigation from MLflow UI\n")

print("4. **Enforce in CI/CD**")
print("   - Make Git logging mandatory")
print("   - Fail pipeline if not in Git repo")
print("   - Ensures consistency\n")

print("5. **Session 4 Connection!**")
print("   - Remember: 'Git is your safety net' (Session 4)")
print("   - MLflow + Git = Complete reproducibility")
print("   - Code version + Data version + Environment = 100% reproducible\n")

print("="*70)
print("\n🎯 PRODUCTION WORKFLOW:\n")

print("1. Developer commits code → Git SHA: abc123")
print("2. Runs experiment → MLflow logs git_commit: abc123")
print("3. Model performs well → Registered in Model Registry (Session 6)")
print("4. Deployed to production")
print("5. Issue found 2 months later")
print("6. Check model in registry → get run_id")
print("7. Check MLflow run → get git_commit: abc123")
print("8. git checkout abc123 → exact code recovered!")
print("9. Debug and fix\n")

print("✅ This is why Git + MLflow integration is critical!")

In [ ]:
# Cell 65: Preparing experiments for Model Registry (Session 6)

print("🚀 Preparing for Session 6: Model Registry\n")
print("="*70)

print("\n📋 MODEL REGISTRY OVERVIEW (Next Session)\n")
print("In Session 6, you'll learn to:")
print("   1. Register models from today's experiments")
print("   2. Version models (v1, v2, v3...)")
print("   3. Add descriptions and metadata")
print("   4. Transition through stages: None → Staging → Production")
print("   5. Serve models via REST API")
print("   6. Implement A/B testing\n")

print("="*70)
print("\n✅ WHAT YOU NEED FROM TODAY'S SESSION:\n")

print("1. **Best Run IDs** (from Cell 53)")
print("   → We saved these to session5_best_runs.json")
print("   → Top 3 models identified")
print("   → Ready to register!\n")

print("2. **Well-Logged Experiments**")
print("   → Parameters logged")
print("   → Metrics tracked")
print("   → Models saved as artifacts")
print("   → Tags for organization\n")

print("3. **Documentation**")
print("   → Run descriptions")
print("   → Dataset versions")
print("   → Git commits")
print("   → Reproducibility info\n")

print("="*70)
print("\n🎯 CHECKLIST: Are Your Experiments Registry-Ready?\n")

checklist = [
    ("Models saved as MLflow artifacts", True),
    ("Comprehensive metrics logged", True),
    ("All hyperparameters recorded", True),
    ("Tags for organization", True),
    ("Dataset version tracked", True),
    ("Git commit logged", False),  # May not be in Git
    ("Run descriptions added", False),  # Optional but recommended
    ("Best models identified", True),
]

for item, required in checklist:
    status = "✅" if required else "📋"
    req_text = "(Required)" if required else "(Recommended)"
    print(f"   {status} {item:40s} {req_text}")

print("\n💡 If you have all ✅ items, you're ready for Session 6!\n")

print("="*70)
print("\n📖 SESSION 6 PREVIEW: The Model Lifecycle\n")

lifecycle = '''
┌──────────────────────────────────────────────────────────┐
│                   MODEL LIFECYCLE                        │
├──────────────────────────────────────────────────────────┤
│                                                          │
│  1. TRAINING (Today - Session 5)                         │
│     └─> Track experiments with MLflow                    │
│     └─> Identify best models                             │
│                                                          │
│  2. REGISTRATION (Session 6)                             │
│     └─> Register model: "customer_churn_classifier"      │
│     └─> Version: v1, v2, v3...                           │
│     └─> Add descriptions and tags                        │
│                                                          │
│  3. STAGING (Session 6)                                  │
│     └─> Transition to 'Staging'                          │
│     └─> Run validation tests                             │
│     └─> Team review and approval                         │
│                                                          │
│  4. PRODUCTION (Session 6)                               │
│     └─> Transition to 'Production'                       │
│     └─> Serve via REST API                               │
│     └─> Monitor performance                              │
│                                                          │
│  5. UPDATES (Module 2)                                   │
│     └─> Automated retraining                             │
│     └─> A/B testing                                      │
│     └─> Rollback if needed                               │
│                                                          │
└──────────────────────────────────────────────────────────┘
'''

print(lifecycle)

print("\n" + "="*70)
print("\n🎓 KEY CONCEPTS FOR SESSION 6:\n")

concepts = [
    ("Model Registry", "Central hub for all production models"),
    ("Model Versions", "v1, v2, v3... track model evolution"),
    ("Lifecycle Stages", "None → Staging → Production → Archived"),
    ("Model Serving", "Deploy models as REST APIs"),
    ("A/B Testing", "Compare models in production"),
]

for concept, description in concepts:
    print(f"   📌 {concept:20s}: {description}")

print("\n" + "="*70)
print("\n📝 ACTION ITEMS BEFORE SESSION 6:\n")

print("   1. ✅ Keep session5_best_runs.json safe")
print("   2. ✅ Review your top 3 models in MLflow UI")
print("   3. ✅ Ensure MLflow is still running: mlflow ui")
print("   4. 📋 Optional: Add descriptions to your best runs")
print("   5. 📋 Optional: Add more tags for organization")
print("   6. 🎯 Get excited about production deployment!\n")

print("="*70)
print("\n🎉 YOU'RE READY FOR SESSION 6!\n")
print("See you next session where we'll take these tracked")
print("experiments and deploy them to production! 🚀")

---

# 🎉 Session 5 Complete: Advanced MLflow Tracking Mastered!

## 🏆 What You've Accomplished Today

Over the past 3 hours, you've gone from MLflow basics to production-grade tracking skills!

---

### ✅ Part 0: Setup & Verification
- Configured MLflow environment
- Verified starter kit installation
- Loaded datasets
- Launched MLflow UI

### ✅ Part 1: MLflow Components & Foundations
- Understood 4 MLflow components (Tracking, Projects, Models, Registry)
- Reviewed basic tracking patterns
- Learned advanced parameter logging
- Created complete tracking examples

### ✅ Part 2: Autolog vs Manual Logging
- Mastered autolog for sklearn, XGBoost, LightGBM, Keras
- Understood when to use each approach
- Implemented hybrid strategies
- Avoided common mistakes
- Combined autolog with custom metrics

### ✅ Part 3: Advanced Artifacts & Dataset Versioning ⭐
- **Implemented professional dataset versioning**
- Calculated dataset hashes for reproducibility
- Tracked complete data lineage
- Logged preprocessing pipelines
- Organized artifacts professionally
- Created model cards and quality reports

### ✅ Part 4: Hyperparameter Tuning & Nested Runs
- Created parent-child run hierarchies
- Implemented grid search (12 combinations)
- Implemented random search (15 iterations)
- Compared search strategies
- Applied best practices
- Saved top models for Model Registry

### ✅ Part 5: MLflow UI Mastery
- Mastered advanced filter syntax
- Used complex query patterns
- Programmatically searched runs
- Exported data for analysis
- Created visualizations

### ✅ Part 6: Production Patterns
- Team collaboration strategies
- Complete reproducibility checklist
- Git integration
- Experiment organization at scale
- Prepared for Model Registry

---

## 📊 By The Numbers

Today you:
- **Ran 40+ tracked experiments** with proper organization
- **Logged 100+ parameters** across all runs
- **Tracked 50+ metrics** for model evaluation
- **Saved 20+ artifacts** (models, plots, configs)
- **Created nested run hierarchies** for hyperparameter searches
- **Implemented dataset versioning** for reproducibility
- **Integrated Git** for code version tracking
- **Prepared 3 models** for production (Session 6)

**You're now an MLflow power user!** 🚀

---

## 🎯 Key Takeaways

### 1. Organization is Critical
- Nested runs keep hyperparameter searches clean
- Tags and naming conventions enable team collaboration
- Proper structure scales to thousands of experiments

### 2. Reproducibility Requires Planning
- Dataset versioning (hash + metadata)
- Git commit tracking
- Environment logging
- Complete artifact preservation

### 3. Autolog + Manual = Best Approach
- Autolog for framework-standard metrics
- Manual logging for custom business metrics
- Combine both strategically

### 4. MLflow UI is Powerful
- Advanced filtering finds anything instantly
- Programmatic access enables automation
- Export capabilities support analysis

### 5. Think Production From Day 1
- Every experiment should be registry-ready
- Log with your future self in mind
- Team collaboration patterns matter

---

## 🚀 What's Next: Session 6

### Model Registry & Deployment

Next week, you'll take your tracked experiments and:

1. **Register Models** in MLflow Model Registry
   - Create model: "customer_churn_classifier"
   - Version models: v1, v2, v3...
   - Add descriptions and metadata

2. **Manage Lifecycle**
   - Transition: None → Staging → Production
   - Approval workflows
   - Version comparison

3. **Deploy Models**
   - Serve models via REST API
   - Batch prediction pipelines
   - Real-time inference

4. **Production Operations**
   - A/B testing strategies
   - Model rollback procedures
   - Performance monitoring

**Bring your `session5_best_runs.json` file!**

---

## 💼 Real-World Application

The skills you learned today are used daily by ML teams at:
- Google, Netflix, Uber, Airbnb
- Financial services firms
- Healthcare AI companies
- E-commerce platforms
- And thousands of other organizations

You're now equipped with **industry-standard MLOps practices**!

---

## 📚 Continue Learning

### Module 2 Preview: Cloud-Native MLOps

Coming up:
- **Airflow**: Automated ML pipelines
- **Kubernetes**: Distributed training and serving
- **Optuna**: Bayesian hyperparameter optimization
- **Prometheus/Grafana**: Production monitoring
- **Advanced deployment**: Blue-green, canary releases

### Module 3: Cloud & Productionization
- AWS SageMaker integration
- Multi-cloud strategies
- Cost optimization
- Feedback loops and retraining

### Module 4: Agentic AI & LLMOps
- LLM fine-tuning and deployment
- RAG architectures
- LangChain, CrewAI, LangGraph
- Evaluation tools: RAGAS, LangSmith

---

## 🎓 Congratulations!

You've completed **Session 5: Advanced MLflow Tracking**!

**Skills Gained**:
- ✅ Professional experiment tracking
- ✅ Dataset versioning and lineage
- ✅ Hyperparameter tuning strategies
- ✅ MLflow UI power user techniques
- ✅ Production-ready workflows
- ✅ Team collaboration patterns

**You're Ready For**:
- Session 6: Model Registry & Deployment
- Real-world ML projects
- Team collaboration
- Production ML systems

---

### 🌟 Thank You!

Thank you for your dedication to learning MLOps. The skills you've gained today will serve you throughout your ML career.

**Questions?** Review the notebook, experiment with the starter kit, and come prepared for Session 6!

**See you in Session 6!** 🚀

---

### 📝 Final Checklist

Before leaving:
- [ ] Save your notebook
- [ ] Keep `session5_best_runs.json` file
- [ ] MLflow UI still running? (Leave it for practice)
- [ ] Review top 3 models one more time
- [ ] Star the course repository 😊
- [ ] Practice with your own datasets this week!

---

## 🎊 You Did It! 🎊

**From beginner to MLflow expert in 3 hours.**

**That's powerful.** 💪

---

*MLOps with Agentic AI*  
*Model Tracking with MLflow*  
*Creator/Author: Amey Talkatkar
*© 2025 - All Rights Reserved*